In [ ]:
# cell 1
HOP1_SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "properties": {
        "entity_queries": {
            "type": "array",
            "minItems": 2,
            "maxItems": 4,
            "items": {
                "type": "object",
                "additionalProperties": False,
                "properties": {
                    "entity": {
                        "type": "string",
                        "minLength": 1
                    },
                    "query": {
                        "type": "string",
                        "minLength": 8
                    }
                },
                "required": ["entity", "query"]
            }
        }
    },
    "required": ["entity_queries"]
}

In [ ]:
# cell 2
HOP1_PROMPT = """
You are a knowledge-graph traversal agent for multi-hop open-domain question answering.

You are given a question. Your job is NOT to answer the question.
Your job is to decide which anchor entities should be searched in a heterogeneous knowledge graph, and what entity-focused information should be retrieved for each entity.

Knowledge graph structure:
- Node types:
  1. entity nodes
     - Entity nodes represent normalized entities.
  2. chunk nodes
     - Chunk nodes represent Wikipedia text chunks.
- Edge types:
  1. entity --relation--> entity
     - relation edges connect entity -> entity.
     - Directed edge.
     - The relation text is a short natural-language sentence describing the connection between two entities.
     - Each relation edge has a relation_id and a source chunk_id.
  2. entity --fact--> chunk
     - fact edges connect entity -> chunk.
     - Directed edge from an entity to a Wikipedia chunk.
     - The fact text describes what information the chunk contains about that entity.
     - Each fact edge has a fact_id and a chunk_id.

Retrieval behavior:
- Entity names will be normalized later by the system.
- Each entity you output will be searched using lexical BM25, character n-gram BM25, and dense entity FAISS retrieval.
- Each query you output will be embedded and matched against relation-edge texts and fact-edge texts.
- Therefore, each query must be short, specific, self-contained, and focused on exactly one entity.

Task:
Given the question, identify the minimum set of important anchor entities needed to start graph traversal.
You must output at least 2 and at most 4 entities.

For each entity, write one natural-language retrieval query asking what information is needed about that entity in order to answer the original question.

Rules:
- Return JSON only.
- Do not answer the question.
- Do not include explanations.
- Do not include generic concepts as entities unless they are central named concepts in the question.
- Prefer named people, organizations, locations, works, events, dates, awards, and Wikipedia-style titles.
- Prefer canonical Wikipedia-style entity names when the question contains aliases, abbreviations, or alternate surface forms.
- Canonical names, aliases, abbreviations, and alternate surface forms are allowed when they preserve the same intended entity.
  Examples:
  - United States, USA, U.S., United States of America
  - United Kingdom, UK, Britain
  - New York City, NYC
- Use canonical names when they are more likely to match KG entity nodes.
- Use alias-like names when they may improve lexical BM25, character n-gram BM25, or dense FAISS retrieval.
- Avoid duplicate entities when the question contains multiple distinct named entities.
- Near-duplicate aliases are allowed only when they improve retrieval and refer to the same intended entity.
- If the question contains only one explicit named entity, still output at least 2 entity queries:
  1. Use the explicit named entity as the first anchor.
  2. Use a useful alternate surface form, disambiguated title, title-like variant, or clue-based variant of the same entity as the second anchor.
- The second anchor for a single-entity question must not be invented from external knowledge. It must be derived from words in the question or from the entity string itself.
- For single-entity questions, the second anchor is allowed to be a near-duplicate if it improves lexical, character n-gram, or dense entity retrieval.
- If the question is a bridge question, include the starting entity and the likely bridge, target, or clue entity when it is explicitly mentioned or clearly implied by the question.
- If the question is a comparison question, include the compared entities and make the queries focus on the compared attribute.
- Each query must be about only its paired entity.
- The query must ask for evidence directly connected to the paired entity, not for a downstream answer about an unknown entity.
- For bridge questions, the first-hop query should usually ask for the next bridge entity or linking evidence, not the final answer attribute.
- Each query must be written so that it can retrieve relation-edge or fact-edge evidence about the paired entity. Do not simply restate the full original question.
- Each query should make clear what missing evidence is needed.
- Do not use external knowledge.

Question:
{question}

Return JSON strictly matching this schema:
{
  "entity_queries": [
    {
      "entity": "...",
      "query": "..."
    }
  ]
}
"""

In [ ]:
# cell 3
HOP2_HOP3_SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "properties": {
        "selected_chunk_ids": {
        "type": "array",
        "minItems": 0,
        "maxItems": 6,
        "uniqueItems": True,
        "items": {
            "type": "string",
            "minLength": 1
        }
    },
            "selected_path_queries": {
            "type": "array",
            "minItems": 0,
            "maxItems": 5,
            "uniqueItems": True,
            "items": {
                "type": "object",
                "additionalProperties": False,
                "properties": {
                    "path_id": {
                        "type": "string",
                        "minLength": 1
                    },
                    "terminal_entity": {
                        "type": "string",
                        "minLength": 1
                    },
                    "query": {
                        "type": "string",
                        "minLength": 8
                    }
                },
                "required": ["path_id", "terminal_entity", "query"]
            }
        },
        "new_entity_queries": {
            "type": "array",
            "minItems": 0,
            "maxItems": 2,
            "uniqueItems": True,
            "items": {
                "type": "object",
                "additionalProperties": False,
                "properties": {
                    "entity": {
                        "type": "string",
                        "minLength": 1
                    },
                    "query": {
                        "type": "string",
                        "minLength": 8
                    }
                },
                "required": ["entity", "query"]
            }
        }
    },
    "required": [
        "selected_chunk_ids",
        "selected_path_queries",
        "new_entity_queries"
    ]
}

In [ ]:
# cell 4
HOP2_HOP3_PROMPT = """
You are a knowledge-graph traversal agent for multi-hop open-domain question answering.

You are given:
1. The original question.
2. Already selected evidence chunks, if any.
3. Candidate evidence chunks.
4. Candidate one-hop or multi-hop entity paths from the knowledge graph.

Your job is NOT to write the final answer.
Your job is to select the most important evidence chunks and the most important graph paths needed to answer the original question.

Knowledge graph structure:
- Node types:
  1. entity nodes
     - Entity nodes represent normalized entities.
  2. chunk nodes
     - Chunk nodes represent Wikipedia text chunks.
- Edge types:
  1. entity --relation--> entity
     - relation edges connect entity -> entity.
     - Directed edge.
     - The relation text is a short natural-language sentence describing the connection between two entities.
     - Each relation edge has a relation_id and a source chunk_id.
  2. entity --fact--> chunk
     - fact edges connect entity -> chunk.
     - Directed edge from an entity to a Wikipedia chunk.
     - The fact text describes what information the chunk contains about that entity.
     - Each fact edge has a fact_id and a chunk_id.

Traversal goal:
The system will continue traversal after this hop.
Even if some chunks already look useful, you must still choose graph paths that can lead to additional missing evidence.

Chunk selection task:
- Consider both "Already selected evidence chunks" and "Candidate evidence chunks".
- Select the chunk IDs that are most important for answering the original question.
- "Available chunk IDs" means the unique chunk_id values shown in "Already selected evidence chunks" plus "Candidate evidence chunks".
- If there are 6 or more unique available chunk IDs, you MUST select exactly 6 chunk IDs.
- If there are fewer than 6 unique available chunk IDs, you MUST select all available chunk IDs.
- If there are zero available chunk IDs, return an empty selected_chunk_ids array.
- Never repeat a chunk_id to fill the quota.
- Do not duplicate chunk IDs.
- Do not invent chunk IDs.
- Prefer chunks that contain answer-bearing facts, bridge evidence, comparison attributes, dates, locations, affiliations, titles, or disambiguating details.
- Keep supporting bridge chunks even if they do not directly contain the final answer, because they may be necessary for multi-hop reasoning.
- If two chunks are similar but both may support different parts of the reasoning chain, keep both when quota allows.

Path selection task:
- Select the graph paths that are most important for reaching the answer.
- "Candidate path IDs" means the unique path_id values shown in "Candidate graph paths".
- If there are 5 or more candidate path IDs, you MUST select exactly 5 path IDs.
- If there are fewer than 5 candidate path IDs, you MUST select all candidate path IDs.
- If there are zero candidate path IDs, return an empty selected_path_queries array.
- Never repeat a path_id to fill the quota.
- Do not duplicate path IDs.
- Do not invent path IDs.
- For each selected path, write a next-step retrieval query about the terminal entity of that path.
- The query must focus only on the terminal entity and ask what information is needed next.
- The query will be embedded and matched against relation-edge texts and fact-edge texts in the next retrieval step.
- Prefer paths that are likely to lead to missing bridge evidence, comparison evidence, answer-bearing facts, aliases, titles, roles, dates, places, or disambiguating information.

New entity task:
- You may propose at most 2 new entity queries.
- Try to propose 2 new entities whenever the question, chunks, or paths suggest useful additional search targets.
- New entities are useful because they improve lexical BM25, character n-gram BM25, dense entity FAISS retrieval, and later KG traversal.
- A new entity may be:
  1. a canonical name for an entity already implied by the evidence,
  2. an alias or alternate surface form,
  3. an abbreviation expansion or abbreviation contraction,
  4. a Wikipedia-style title variant,
  5. a bridge entity suggested by the current path,
  6. a clue entity needed for comparison or disambiguation.
- Examples of acceptable canonical/alias variants when they preserve the same meaning:
  - United States, USA, U.S., United States of America
  - United Kingdom, UK, Britain
  - New York City, NYC
- Do not propose unrelated entities.
- Do not use external knowledge to guess the final answer.
- If no reasonable new entity exists, return an empty array.
- If only one reasonable new entity exists, return one.
- Canonical names and aliases should preserve the same intended entity, but they do not need to look textually similar.
  For example, "United States" and "USA" are acceptable variants of the same entity.
- Use canonical names, aliases, or abbreviation variants only for entities already mentioned or clearly implied by the question, chunks, or paths.
- Do not introduce a new entity only because it is generally related; it must help retrieve missing evidence for this question.

ID copying rules:
- Copy every chunk_id exactly from the input.
- The path_id values shown in this prompt are short prompt-local IDs such as p1, p2, p3.
- Copy only these short path_id values exactly from the input.
- Do not rewrite, shorten, normalize, compose, or repair IDs.
- Do not copy a relation_id as a path_id.
- Do not create a path_id by combining pieces of other path IDs.
- The selected path_id must be character-for-character identical to one candidate path_id shown in this prompt.
- For terminal_entity, copy the terminal_entity field of the selected path.

Output rules:
- Return JSON only.
- Do not include explanations.
- Do not include markdown.
- Do not answer the question.
- Do not output any stopping signal.

Original question:
{question}

Already selected evidence chunks:
{selected_chunks}

Candidate evidence chunks:
{candidate_chunks}

Candidate graph paths:
{candidate_paths}

Input format notes:
- Each chunk is provided with chunk_id, title, and text.
- Each graph path is provided with path_id, path, terminal_entity, and source_chunk_id.
- selected_chunk_ids may include IDs from already selected evidence chunks and candidate evidence chunks.
- selected_path_queries may include only IDs from candidate graph paths.

Return JSON strictly matching this schema:
{
  "selected_chunk_ids": [
    "..."
  ],
  "selected_path_queries": [
    {
      "path_id": "...",
      "terminal_entity": "...",
      "query": "..."
    }
  ],
  "new_entity_queries": [
    {
      "entity": "...",
      "query": "..."
    }
  ]
}
"""

In [ ]:
# cell 5
HOP4_SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "properties": {
        "selected_chunk_ids": {
            "type": "array",
            "minItems": 0,
            "maxItems": 6,
            "uniqueItems": True,
            "items": {
                "type": "string",
                "minLength": 1
            }
        },
        "selected_path_ids": {
            "type": "array",
            "minItems": 0,
            "maxItems": 5,
            "uniqueItems": True,
            "items": {
                "type": "string",
                "minLength": 1
            }
        }
    },
    "required": [
        "selected_chunk_ids",
        "selected_path_ids"
    ]
}

In [ ]:
# cell 6
HOP4_PROMPT = """
You are a knowledge-graph traversal agent for multi-hop open-domain question answering.

You are given:
1. The original question.
2. Already selected evidence chunks from previous hops.
3. Candidate evidence chunks from the final retrieval step.
4. Candidate graph paths from the knowledge graph.

This is the final traversal hop.
Your job is NOT to write the final answer.
Your job is to select the most important final evidence chunks and the most important path IDs needed to support answering the original question.

Knowledge graph structure:
- Node types:
  1. entity nodes
     - Entity nodes represent normalized entities.
  2. chunk nodes
     - Chunk nodes represent Wikipedia text chunks.
- Edge types:
  1. entity --relation--> entity
     - relation edges connect entity -> entity.
     - Directed edge.
     - The relation text is a short natural-language sentence describing the connection between two entities.
     - Each relation edge has a relation_id and a source chunk_id.
  2. entity --fact--> chunk
     - fact edges connect entity -> chunk.
     - Directed edge from an entity to a Wikipedia chunk.
     - The fact text describes what information the chunk contains about that entity.
     - Each fact edge has a fact_id and a chunk_id.

Final-hop chunk selection task:
- Consider both "Already selected evidence chunks" and "Candidate evidence chunks".
- Select the chunk IDs that are most important for answering the original question.
- "Available chunk IDs" means the unique chunk_id values shown in "Already selected evidence chunks" plus "Candidate evidence chunks".
- If there are 6 or more unique available chunk IDs, you MUST select exactly 6 chunk IDs.
- If there are fewer than 6 unique available chunk IDs, you MUST select all available chunk IDs.
- If there are zero available chunk IDs, return an empty selected_chunk_ids array.
- Never repeat a chunk_id to fill the quota.
- Do not duplicate chunk IDs.
- Do not invent chunk IDs.
- Prefer chunks that contain explicit answer-bearing facts.
- Also keep bridge evidence, comparison evidence, dates, locations, affiliations, titles, attributes, aliases, or disambiguating details when they are needed to justify the answer.
- The final answer generator will use the selected chunks, so do not be too conservative.
- If a chunk supports a required intermediate step, select it even if it does not directly contain the final answer.

Final-hop path selection task:
- Select path IDs that are most important for evidence tracing and reasoning support.
- "Candidate path IDs" means the unique path_id values shown in "Candidate graph paths".
- If there are 5 or more candidate path IDs, you MUST select exactly 5 path IDs.
- If there are fewer than 5 candidate path IDs, you MUST select all candidate path IDs.
- If there are zero candidate path IDs, return an empty selected_path_ids array.
- Never repeat a path_id to fill the quota.
- Do not duplicate path IDs.
- Do not invent path IDs.
- Prefer paths that connect the question entities to bridge entities, comparison entities, titles, roles, dates, places, or answer-bearing facts.
- Prefer paths whose source_chunk_id corresponds to selected or useful chunks.
- Selected paths do not need new queries in this final hop.

ID copying rules:
- Copy every chunk_id exactly from the input.
- The path_id values shown in this prompt are short prompt-local IDs such as p1, p2, p3.
- Copy only these short path_id values exactly from the input.
- Do not rewrite, shorten, normalize, compose, or repair IDs.
- Do not copy a relation_id as a path_id.
- Do not create a path_id by combining pieces of other path IDs.
- The selected path_id must be character-for-character identical to one candidate path_id shown in this prompt.

Output rules:
- Return JSON only.
- Do not include explanations.
- Do not include markdown.
- Do not answer the question.
- Do not write new queries.
- Do not propose new entities.
- Do not decide whether to continue.

Original question:
{question}

Already selected evidence chunks:
{selected_chunks}

Candidate evidence chunks:
{candidate_chunks}

Candidate graph paths:
{candidate_paths}

Input format notes:
- Each chunk is provided with chunk_id, title, and text.
- Each graph path is provided with path_id, path, terminal_entity, source_chunk_id, and path_depth.
- selected_chunk_ids may include IDs from already selected evidence chunks and candidate evidence chunks.
- selected_path_ids may include only IDs from candidate graph paths.

Return JSON strictly matching this schema:
{
  "selected_chunk_ids": [
    "..."
  ],
  "selected_path_ids": [
    "..."
  ]
}
"""

In [ ]:
# cell 7
# Install runtime dependencies.

!pip -q install -U uv

# Core runtime packages.
!uv pip install --system -U openai requests tqdm jsonschema psutil numpy accelerate safetensors scipy joblib faiss-cpu torch-geometric

# Remove optional packages that may break model imports.
!uv pip uninstall --system -y torchcodec torchvision torchaudio sentence-transformers || true

# Transformers for embedding and reranker.
!uv pip install --system -U "transformers>=4.51.0"

# vLLM nightly for CUDA 13 / Blackwell.
!uv pip install --system -U vllm --torch-backend=cu130 --extra-index-url https://wheels.vllm.ai/nightly/cu130

# Optional fallback if the cu130 line fails:
# !uv pip install --system -U vllm --torch-backend=auto --extra-index-url https://wheels.vllm.ai/nightly

import sys
import importlib.metadata as md

import torch
import vllm
import transformers

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print("vLLM:", vllm.__version__)
print("transformers:", transformers.__version__)

for pkg in ["torchcodec", "torchvision", "torchaudio", "sentence-transformers"]:
    try:
        print(pkg + ":", md.version(pkg))
    except Exception:
        print(pkg + ": not installed")

Using Python 3.12.13 environment at: /usr
Resolved 81 packages in 160ms
Prepared 8 packages in 0.37ms
Uninstalled 8 packages in 124ms
Installed 8 packages in 108ms
 - numpy==2.3.5
 + numpy==2.4.6
 - nvidia-cublas==13.1.0.3
 + nvidia-cublas==13.1.1.3
 - nvidia-cudnn-cu13==9.19.0.56
 + nvidia-cudnn-cu13==9.20.0.48
 - nvidia-cusparselt-cu13==0.8.0
 + nvidia-cusparselt-cu13==0.8.1
 - nvidia-nccl-cu13==2.28.9
 + nvidia-nccl-cu13==2.29.7
 - setuptools==80.10.2
 + setuptools==81.0.0
 - torch==2.11.0+cu130
 + torch==2.12.1
 - triton==3.6.0
 + triton==3.7.1
Using Python 3.12.13 environment at: /usr
Uninstalled 2 packages in 48ms
 - torchaudio==2.11.0+cu130
 - torchvision==0.26.0+cu130
Using Python 3.12.13 environment at: /usr
Resolved 27 packages in 103ms
Checked 27 packages in 0.29ms
Using Python 3.12.13 environment at: /usr
Resolved 189 packages in 9.18s
Prepared 10 packages in 19ms
Uninstalled 8 packages in 110ms
Installed 10 packages in 111ms
 - numpy==2.4.6
 + numpy==2.3.5
 - nvidia-cublas

In [ ]:
# cell 8
# Imports and global configuration.

import os
import re
import gc
import html
import json
import time
import shlex
import shutil
import psutil
import joblib
import faiss
import unicodedata
import subprocess

from pathlib import Path
from collections import defaultdict

import numpy as np
import scipy.sparse as sp

import torch
import torch.nn.functional as F
from torch import Tensor

from tqdm.auto import tqdm
from jsonschema import validate
from openai import OpenAI
from torch_geometric.data import HeteroData

os.environ["TOKENIZERS_PARALLELISM"] = "false"

DATASET = "2wikimultihopqa"

# Drive paths are defined here, but Drive folders are created after mount in cell 9.
FINAL_PROJECT_DIR = Path("/content/drive/MyDrive/final_project")
IDEA_DIR = FINAL_PROJECT_DIR / "idea_1"

DRIVE_DEV_PATH = FINAL_PROJECT_DIR / "2wikimultihopqa_dev_2020wiki_1000_converted.json"
DRIVE_CONSTRUCT_DIR = IDEA_DIR / "kg" / DATASET / "kg_construct"

# Local runtime files stay on Colab local disk.
# The actual KG/index/embedding objects will be loaded into RAM in later cells.
LOCAL_RUNTIME_DIR = Path("/content/2wikimultihopqa_traversal_runtime")
LOCAL_DATA_DIR = LOCAL_RUNTIME_DIR / "data"
LOCAL_CONSTRUCT_DIR = LOCAL_RUNTIME_DIR / "kg_construct"

LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_CONSTRUCT_DIR.mkdir(parents=True, exist_ok=True)

# Do not create this before Drive mount.
EVIDENCE_DIR = FINAL_PROJECT_DIR / "ablation" / "3llm_call_(hop)"
EVIDENCE_OUTPUT_PATH = EVIDENCE_DIR / "2wikimultihopqa_dev_2020wiki_1000_traversal_evidence.json"
EVIDENCE_TMP_PATH = EVIDENCE_OUTPUT_PATH.with_suffix(".tmp.json")

LLM_MODEL_NAME = "Qwen/Qwen3.5-27B"
PORT = 8000
BASE_URL = f"http://localhost:{PORT}/v1"

# Model runtime settings.
MAX_MODEL_LEN = 32768
GPU_MEMORY_UTILIZATION = 0.70

MAX_NUM_SEQS = 1
MAX_NUM_BATCHED_TOKENS = 32768

SERVER_LOG_PATH = Path("/content/vllm_server.log")
SERVER_PID_PATH = Path("/content/vllm_server.pid")

EMBED_MODEL_NAME = "Qwen/Qwen3-Embedding-8B"
RERANKER_MODEL_NAME = "Qwen/Qwen3-Reranker-4B"

# No manual truncation for embedding/reranker inputs.
EMBED_MAX_LENGTH = None
RERANKER_MAX_LENGTH = None

TRAVERSAL_CONFIG = {
    "hop1_relation_top_k_per_entity": 3,
    "hop1_fact_top_k_per_entity": 4,
    "path_relation_top_k": 3,
    "path_fact_top_k": 3,
    "new_entity_top_k_per_method": 2,
    "new_entity_relation_top_k": 2,
    "new_entity_fact_top_k": 3,
    "path_expansion_final_chunk_top_k": 3,
    "new_entity_final_chunk_top_k_per_group": 2,
    "max_final_chunks": 6,
    "max_final_paths": 5,
}

REQUIRED_PROMPT_OBJECTS = [
    "HOP1_SCHEMA",
    "HOP1_PROMPT",
    "HOP2_HOP3_SCHEMA",
    "HOP2_HOP3_PROMPT",
    "HOP4_SCHEMA",
    "HOP4_PROMPT",
]

missing_prompt_objects = [name for name in REQUIRED_PROMPT_OBJECTS if name not in globals()]
if missing_prompt_objects:
    raise RuntimeError(f"Missing prompt/schema objects from earlier cells: {missing_prompt_objects}")

print("Runtime dir:", LOCAL_RUNTIME_DIR)
print("Local construct dir:", LOCAL_CONSTRUCT_DIR)
print("Evidence output path:", EVIDENCE_OUTPUT_PATH)
print("LLM max model len:", MAX_MODEL_LEN)
print("vLLM GPU utilization:", GPU_MEMORY_UTILIZATION)

Runtime dir: /content/2wikimultihopqa_traversal_runtime
Local construct dir: /content/2wikimultihopqa_traversal_runtime/kg_construct
Evidence output path: /content/drive/MyDrive/final_project/ablation/3llm_call_(hop)/2wikimultihopqa_dev_2020wiki_1000_traversal_evidence.json
LLM max model len: 32768
vLLM GPU utilization: 0.7


In [ ]:
# cell 9
# Mount Drive, copy files to local runtime, and create evidence output file.

from google.colab import drive

MOUNTPOINT = Path("/content/drive")

# Do not delete /content/drive. Just mount safely.
drive.mount(str(MOUNTPOINT), force_remount=True)

# Re-create Drive-dependent paths after mount to be safe.
FINAL_PROJECT_DIR = Path("/content/drive/MyDrive/final_project")
IDEA_DIR = FINAL_PROJECT_DIR / "idea_1"

DRIVE_DEV_PATH = FINAL_PROJECT_DIR / "2wikimultihopqa_dev_2020wiki_1000_converted.json"
DRIVE_CONSTRUCT_DIR = IDEA_DIR / "kg" / DATASET / "kg_construct"

EVIDENCE_DIR = FINAL_PROJECT_DIR / "ablation" / "3llm_call_(hop)"
EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)

EVIDENCE_OUTPUT_PATH = EVIDENCE_DIR / "2wikimultihopqa_dev_2020wiki_1000_traversal_evidence.json"
EVIDENCE_TMP_PATH = EVIDENCE_OUTPUT_PATH.with_suffix(".tmp.json")

DRIVE_FILES = {
    "dev_json": DRIVE_DEV_PATH,
    "hetero_kg_pt": DRIVE_CONSTRUCT_DIR / "2wikimultihopqa_hetero_kg.pt",
    "kg_meta_json": DRIVE_CONSTRUCT_DIR / "2wikimultihopqa_hetero_kg_meta.json",
    "entity_nodes_jsonl": DRIVE_CONSTRUCT_DIR / "2wikimultihopqa_entity_nodes_for_kg.jsonl",
    "chunk_text_catalog_jsonl": DRIVE_CONSTRUCT_DIR / "2wikimultihopqa_chunk_text_catalog.jsonl",
    "relation_edges_jsonl": DRIVE_CONSTRUCT_DIR / "2wikimultihopqa_relation_edges_for_kg.jsonl",
    "fact_edges_jsonl": DRIVE_CONSTRUCT_DIR / "2wikimultihopqa_fact_edges_for_kg.jsonl",
    "chunk_node_id_to_chunk_id_json": DRIVE_CONSTRUCT_DIR / "2wikimultihopqa_chunk_node_id_to_chunk_id.json",
    "token_bm25_joblib": DRIVE_CONSTRUCT_DIR / "2wikimultihopqa_entity_token_bm25.joblib",
    "char3_bm25_joblib": DRIVE_CONSTRUCT_DIR / "2wikimultihopqa_entity_char3_bm25.joblib",
    "entity_faiss": DRIVE_CONSTRUCT_DIR / "2wikimultihopqa_entities.indexflatip.idmap.faiss",
    "relation_embeddings_npy": DRIVE_CONSTRUCT_DIR / "relation_embeddings.npy",
    "fact_embeddings_npy": DRIVE_CONSTRUCT_DIR / "fact_embeddings.npy",
}

for name, path in DRIVE_FILES.items():
    if not path.exists():
        raise FileNotFoundError(f"Required file not found for {name}: {path}")

LOCAL_FILES = {
    "dev_json": LOCAL_DATA_DIR / DRIVE_DEV_PATH.name,
    "hetero_kg_pt": LOCAL_CONSTRUCT_DIR / "2wikimultihopqa_hetero_kg.pt",
    "kg_meta_json": LOCAL_CONSTRUCT_DIR / "2wikimultihopqa_hetero_kg_meta.json",
    "entity_nodes_jsonl": LOCAL_CONSTRUCT_DIR / "2wikimultihopqa_entity_nodes_for_kg.jsonl",
    "chunk_text_catalog_jsonl": LOCAL_CONSTRUCT_DIR / "2wikimultihopqa_chunk_text_catalog.jsonl",
    "relation_edges_jsonl": LOCAL_CONSTRUCT_DIR / "2wikimultihopqa_relation_edges_for_kg.jsonl",
    "fact_edges_jsonl": LOCAL_CONSTRUCT_DIR / "2wikimultihopqa_fact_edges_for_kg.jsonl",
    "chunk_node_id_to_chunk_id_json": LOCAL_CONSTRUCT_DIR / "2wikimultihopqa_chunk_node_id_to_chunk_id.json",
    "token_bm25_joblib": LOCAL_CONSTRUCT_DIR / "2wikimultihopqa_entity_token_bm25.joblib",
    "char3_bm25_joblib": LOCAL_CONSTRUCT_DIR / "2wikimultihopqa_entity_char3_bm25.joblib",
    "entity_faiss": LOCAL_CONSTRUCT_DIR / "2wikimultihopqa_entities.indexflatip.idmap.faiss",
    "relation_embeddings_npy": LOCAL_CONSTRUCT_DIR / "relation_embeddings.npy",
    "fact_embeddings_npy": LOCAL_CONSTRUCT_DIR / "fact_embeddings.npy",
}

def file_is_same_size(src: Path, dst: Path) -> bool:
    # Check whether a local copy is already complete.
    return dst.exists() and dst.stat().st_size == src.stat().st_size

def copy_file_to_local(src: Path, dst: Path) -> None:
    # Copy with a temp file to avoid partial outputs.
    dst.parent.mkdir(parents=True, exist_ok=True)

    if file_is_same_size(src, dst):
        return

    tmp = dst.with_name(dst.name + ".tmp")

    if tmp.exists():
        tmp.unlink()

    shutil.copy2(src, tmp)
    os.replace(tmp, dst)

for name in tqdm(DRIVE_FILES.keys(), desc="Copying Drive files to local runtime"):
    copy_file_to_local(DRIVE_FILES[name], LOCAL_FILES[name])

for name, path in LOCAL_FILES.items():
    if not path.exists():
        raise FileNotFoundError(f"Local copy missing for {name}: {path}")

# Create the output JSON file on real Google Drive.
if not EVIDENCE_OUTPUT_PATH.exists():
    EVIDENCE_OUTPUT_PATH.write_text("[]", encoding="utf-8")

print("All required files are available locally.")
print("Evidence directory:", EVIDENCE_DIR)
print("Evidence file:", EVIDENCE_OUTPUT_PATH)
print("Evidence file exists:", EVIDENCE_OUTPUT_PATH.exists())

Mounted at /content/drive


Copying Drive files to local runtime:   0%|          | 0/13 [00:00<?, ?it/s]

All required files are available locally.
Evidence directory: /content/drive/MyDrive/final_project/ablation/3llm_call_(hop)
Evidence file: /content/drive/MyDrive/final_project/ablation/3llm_call_(hop)/2wikimultihopqa_dev_2020wiki_1000_traversal_evidence.json
Evidence file exists: True


In [ ]:
# cell 10
# Load dev questions.

with open(LOCAL_FILES["dev_json"], "r", encoding="utf-8") as f:
    dev_raw = json.load(f)

def get_dataset_items(data):
    # Support common JSON layouts.
    if isinstance(data, list):
        return data

    if isinstance(data, dict):
        for key in ["data", "examples", "items"]:
            if key in data and isinstance(data[key], list):
                return data[key]

    raise RuntimeError("Unsupported dev JSON structure.")

dev_items = get_dataset_items(dev_raw)

question_records = []

for idx, item in enumerate(dev_items):
    question = item.get("question") or item.get("Question") or item.get("query") or item.get("Query")
    answer = item.get("answer") or item.get("Answer")
    q_type = item.get("type") or item.get("Type")
    supports = item.get("supports", [])

    if not question:
        raise RuntimeError(f"Question field not found at index {idx}. Available keys: {list(item.keys())}")

    question_records.append({
        "source_index": idx,
        "type": q_type,
        "question": question,
        "answer": answer,
        "supports": supports,
    })

print("Loaded questions:", len(question_records))
print("First question:", question_records[0]["question"])

Loaded questions: 1000
First question: Are North Marion High School (Oregon) and Seoul High School both located in the same country?


In [ ]:
# cell 11
# Load heterogeneous KG and metadata into RAM.

kg = torch.load(
    LOCAL_FILES["hetero_kg_pt"],
    map_location="cpu",
    weights_only=False,
)

with open(LOCAL_FILES["kg_meta_json"], "r", encoding="utf-8") as f:
    kg_meta = json.load(f)

with open(LOCAL_FILES["chunk_node_id_to_chunk_id_json"], "r", encoding="utf-8") as f:
    chunk_node_id_to_chunk_id = json.load(f)

chunk_id_to_chunk_node_id = {
    str(chunk_id): int(chunk_node_id)
    for chunk_node_id, chunk_id in chunk_node_id_to_chunk_id.items()
}

print("KG loaded.")
print(kg)
print("Node types:", kg.node_types)
print("Edge types:", kg.edge_types)
print("Chunk node mapping size:", len(chunk_node_id_to_chunk_id))

KG loaded.
HeteroData(
  entity={
    num_nodes=146864,
    entity_id=[146864],
  },
  chunk={
    num_nodes=12685,
    chunk_node_id=[12685],
  },
  (entity, relation, entity)={
    edge_index=[2, 247252],
    relation_id=[247252],
    chunk_node_id=[247252],
  },
  (entity, fact, chunk)={
    edge_index=[2, 184001],
    fact_id=[184001],
  },
  (chunk, next_chunk, chunk)={ edge_index=[2, 6855] },
  (chunk, prev_chunk, chunk)={ edge_index=[2, 6855] }
)
Node types: ['entity', 'chunk']
Edge types: [('entity', 'relation', 'entity'), ('entity', 'fact', 'chunk'), ('chunk', 'next_chunk', 'chunk'), ('chunk', 'prev_chunk', 'chunk')]
Chunk node mapping size: 12685


In [ ]:
# cell 12
# Load entity, chunk, relation, and fact sidecars into RAM.

id_to_entity = {}
entity_to_id = {}

with open(LOCAL_FILES["entity_nodes_jsonl"], "r", encoding="utf-8") as f:
    for line in tqdm(f, desc="Loading entity nodes"):
        if not line.strip():
            continue
        rec = json.loads(line)
        entity_id = int(rec["entity_id"])
        entity = rec["entity"]
        id_to_entity[entity_id] = entity
        entity_to_id[entity] = entity_id

chunk_text_by_node_id = {}
chunk_text_by_chunk_id = {}

with open(LOCAL_FILES["chunk_text_catalog_jsonl"], "r", encoding="utf-8") as f:
    for line in tqdm(f, desc="Loading chunk text catalog"):
        if not line.strip():
            continue
        rec = json.loads(line)
        node_id = int(rec["chunk_node_id"])
        chunk_id = str(rec["chunk_id"])
        chunk_text_by_node_id[node_id] = rec
        chunk_text_by_chunk_id[chunk_id] = rec

relation_record_by_id = {}

with open(LOCAL_FILES["relation_edges_jsonl"], "r", encoding="utf-8") as f:
    for line in tqdm(f, desc="Loading relation edges"):
        if not line.strip():
            continue
        rec = json.loads(line)
        relation_record_by_id[int(rec["relation_id"])] = rec

fact_record_by_id = {}

with open(LOCAL_FILES["fact_edges_jsonl"], "r", encoding="utf-8") as f:
    for line in tqdm(f, desc="Loading fact edges"):
        if not line.strip():
            continue
        rec = json.loads(line)
        fact_record_by_id[int(rec["fact_id"])] = rec

print("Entities:", len(id_to_entity))
print("Chunks:", len(chunk_text_by_chunk_id))
print("Relation records:", len(relation_record_by_id))
print("Fact records:", len(fact_record_by_id))

Loading entity nodes: 0it [00:00, ?it/s]

Loading chunk text catalog: 0it [00:00, ?it/s]

Loading relation edges: 0it [00:00, ?it/s]

Loading fact edges: 0it [00:00, ?it/s]

Entities: 146864
Chunks: 12685
Relation records: 247252
Fact records: 184001


In [ ]:
# cell 13
# Load retrieval indexes and edge embeddings into RAM.

token_bm25_index = joblib.load(LOCAL_FILES["token_bm25_joblib"])
char3_bm25_index = joblib.load(LOCAL_FILES["char3_bm25_joblib"])

entity_faiss_index = faiss.read_index(str(LOCAL_FILES["entity_faiss"]))

# Load fully into RAM. Do not use mmap.
relation_embeddings = np.load(LOCAL_FILES["relation_embeddings_npy"], mmap_mode=None)
fact_embeddings = np.load(LOCAL_FILES["fact_embeddings_npy"], mmap_mode=None)

relation_embeddings = relation_embeddings.astype(np.float32, copy=False)
fact_embeddings = fact_embeddings.astype(np.float32, copy=False)

if not relation_embeddings.flags["C_CONTIGUOUS"]:
    relation_embeddings = np.ascontiguousarray(relation_embeddings, dtype=np.float32)

if not fact_embeddings.flags["C_CONTIGUOUS"]:
    fact_embeddings = np.ascontiguousarray(fact_embeddings, dtype=np.float32)

print("Token BM25 entities:", token_bm25_index["num_entities"])
print("Char3 BM25 entities:", char3_bm25_index["num_entities"])
print("Entity FAISS ntotal:", entity_faiss_index.ntotal)
print("Relation embeddings:", relation_embeddings.shape, relation_embeddings.dtype)
print("Fact embeddings:", fact_embeddings.shape, fact_embeddings.dtype)

ram = psutil.virtual_memory()
print(f"RAM used: {ram.used / (1024 ** 3):.2f} GB / {ram.total / (1024 ** 3):.2f} GB")

Token BM25 entities: 146864
Char3 BM25 entities: 146864
Entity FAISS ntotal: 146864
Relation embeddings: (247252, 4096) float32
Fact embeddings: (184001, 4096) float32
RAM used: 13.36 GB / 176.88 GB


In [ ]:
# cell 14
# Build adjacency lists for fast traversal.

rel_edge_index = kg[("entity", "relation", "entity")].edge_index
rel_ids_tensor = kg[("entity", "relation", "entity")].relation_id
rel_chunk_node_ids_tensor = kg[("entity", "relation", "entity")].chunk_node_id

fact_edge_index = kg[("entity", "fact", "chunk")].edge_index
fact_ids_tensor = kg[("entity", "fact", "chunk")].fact_id

rel_src_np = rel_edge_index[0].cpu().numpy().astype(np.int64, copy=False)
rel_dst_np = rel_edge_index[1].cpu().numpy().astype(np.int64, copy=False)
rel_ids_np = rel_ids_tensor.cpu().numpy().astype(np.int64, copy=False)
rel_chunk_node_ids_np = rel_chunk_node_ids_tensor.cpu().numpy().astype(np.int64, copy=False)

fact_src_np = fact_edge_index[0].cpu().numpy().astype(np.int64, copy=False)
fact_dst_chunk_np = fact_edge_index[1].cpu().numpy().astype(np.int64, copy=False)
fact_ids_np = fact_ids_tensor.cpu().numpy().astype(np.int64, copy=False)

relation_out_positions_by_entity = defaultdict(list)
for pos, src_entity_id in enumerate(tqdm(rel_src_np, desc="Building relation adjacency")):
    relation_out_positions_by_entity[int(src_entity_id)].append(pos)

fact_positions_by_entity = defaultdict(list)
for pos, src_entity_id in enumerate(tqdm(fact_src_np, desc="Building fact adjacency")):
    fact_positions_by_entity[int(src_entity_id)].append(pos)

# Optional chunk adjacency for later use.
next_chunk_edge_index = kg[("chunk", "next_chunk", "chunk")].edge_index
prev_chunk_edge_index = kg[("chunk", "prev_chunk", "chunk")].edge_index

next_chunk_positions_by_node = defaultdict(list)
prev_chunk_positions_by_node = defaultdict(list)

next_src_np = next_chunk_edge_index[0].cpu().numpy().astype(np.int64, copy=False)
next_dst_np = next_chunk_edge_index[1].cpu().numpy().astype(np.int64, copy=False)
prev_src_np = prev_chunk_edge_index[0].cpu().numpy().astype(np.int64, copy=False)
prev_dst_np = prev_chunk_edge_index[1].cpu().numpy().astype(np.int64, copy=False)

for pos, src_node_id in enumerate(next_src_np):
    next_chunk_positions_by_node[int(src_node_id)].append(pos)

for pos, src_node_id in enumerate(prev_src_np):
    prev_chunk_positions_by_node[int(src_node_id)].append(pos)

print("Relation adjacency entities:", len(relation_out_positions_by_entity))
print("Fact adjacency entities:", len(fact_positions_by_entity))
print("Next-chunk adjacency nodes:", len(next_chunk_positions_by_node))
print("Prev-chunk adjacency nodes:", len(prev_chunk_positions_by_node))

Building relation adjacency:   0%|          | 0/247252 [00:00<?, ?it/s]

Building fact adjacency:   0%|          | 0/184001 [00:00<?, ?it/s]

Relation adjacency entities: 62532
Fact adjacency entities: 106579
Next-chunk adjacency nodes: 6855
Prev-chunk adjacency nodes: 6855


In [ ]:
# cell 15
# Normalization and BM25 helpers.

def normalize_entity_text(text):
    # Match KG entity normalization.
    if text is None:
        return ""

    text = str(text)
    text = html.unescape(text)
    text = unicodedata.normalize("NFKC", text)

    replacements = {
        "\u2018": "'",
        "\u2019": "'",
        "\u201c": '"',
        "\u201d": '"',
        "\u2013": "-",
        "\u2014": "-",
        "\u2212": "-",
        "\u00a0": " ",
    }

    for src, dst in replacements.items():
        text = text.replace(src, dst)

    text = re.sub(r"\s+", " ", text).strip()
    text = text.strip(" \t\r\n\"'`")
    text = text.lower()
    return text

TOKEN_RE = re.compile(r"[a-z0-9]+(?:'[a-z0-9]+)?")

def token_analyzer(text):
    # Token BM25 analyzer.
    text = normalize_entity_text(text)
    return TOKEN_RE.findall(text)

def char3_analyzer(text):
    # Character 3-gram BM25 analyzer.
    text = normalize_entity_text(text)
    if not text:
        return []
    padded = f" {text} "
    if len(padded) < 3:
        return [padded]
    return [padded[i:i + 3] for i in range(len(padded) - 2)]

def get_query_terms(query, index_type):
    # Select BM25 analyzer.
    if index_type == "token_bm25":
        return token_analyzer(query)
    if index_type == "char3_bm25":
        return char3_analyzer(query)
    raise ValueError(f"Unknown index_type: {index_type}")

def search_entity_bm25(query, index_payload, top_k=10):
    # Search entity BM25 index.
    vocab = index_payload["vocab"]
    matrix = index_payload["bm25_matrix"]
    entities = index_payload["entities"]
    index_type = index_payload["index_type"]

    terms = get_query_terms(query, index_type)
    term_ids = sorted({vocab[t] for t in terms if t in vocab})

    if not term_ids:
        return []

    scores = np.asarray(matrix[:, term_ids].sum(axis=1)).ravel()
    top_k = min(int(top_k), len(scores))

    if top_k <= 0:
        return []

    candidate_ids = np.argpartition(-scores, top_k - 1)[:top_k]
    candidate_ids = candidate_ids[np.argsort(-scores[candidate_ids])]

    results = []
    for entity_id in candidate_ids:
        score = float(scores[entity_id])
        if score <= 0:
            continue
        results.append({
            "entity_id": int(entity_id),
            "entity": entities[int(entity_id)],
            "score": score,
        })

    return results

print("Normalization and BM25 helpers are ready.")

Normalization and BM25 helpers are ready.


In [ ]:
# cell 16
# Start Qwen vLLM server.

def kill_process_tree(pid):
    # Kill a process and its children.
    try:
        parent = psutil.Process(int(pid))
        for child in parent.children(recursive=True):
            try:
                child.kill()
            except Exception:
                pass
        parent.kill()
        parent.wait(timeout=10)
    except Exception:
        pass

# Stop old PID.
if SERVER_PID_PATH.exists():
    old_pid = SERVER_PID_PATH.read_text().strip()
    if old_pid:
        kill_process_tree(old_pid)

# Stop leftover vLLM serve processes.
for p in psutil.process_iter(["pid", "name", "cmdline"]):
    try:
        cmdline = " ".join(p.info.get("cmdline") or [])
        if "vllm" in cmdline and "serve" in cmdline:
            kill_process_tree(p.info["pid"])
    except Exception:
        pass

time.sleep(3)

cmd = [
    "vllm", "serve", LLM_MODEL_NAME,
    "--host", "0.0.0.0",
    "--port", str(PORT),
    "--max-model-len", str(MAX_MODEL_LEN),
    "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION),
    "--language-model-only",
    "--reasoning-parser", "qwen3",
    "--default-chat-template-kwargs", '{"enable_thinking": false}',
    "--max-num-seqs", str(MAX_NUM_SEQS),
    "--max-num-batched-tokens", str(MAX_NUM_BATCHED_TOKENS),
    "--enable-prefix-caching",
    "--generation-config", "vllm",
    "--dtype", "bfloat16",
    "--trust-remote-code",
]

server_env = os.environ.copy()
server_env["VLLM_USE_FLASHINFER_SAMPLER"] = "0"
server_env["VLLM_MAIN_CUDA_VERSION"] = "13.0"
server_env["TORCH_CUDA_ARCH_LIST"] = "12.0"

print("Command:")
print(" ".join(shlex.quote(x) for x in cmd))

SERVER_LOG_PATH.write_text("", encoding="utf-8")
log_file = open(SERVER_LOG_PATH, "w", encoding="utf-8")

proc = subprocess.Popen(
    cmd,
    stdout=log_file,
    stderr=subprocess.STDOUT,
    text=True,
    env=server_env,
)

SERVER_PID_PATH.write_text(str(proc.pid))

print("Started vLLM server.")
print("PID:", proc.pid)
print("Log:", SERVER_LOG_PATH)

Command:
vllm serve Qwen/Qwen3.5-27B --host 0.0.0.0 --port 8000 --max-model-len 32768 --gpu-memory-utilization 0.7 --language-model-only --reasoning-parser qwen3 --default-chat-template-kwargs '{"enable_thinking": false}' --max-num-seqs 1 --max-num-batched-tokens 32768 --enable-prefix-caching --generation-config vllm --dtype bfloat16 --trust-remote-code
Started vLLM server.
PID: 6085
Log: /content/vllm_server.log


In [ ]:
# cell 17
# Wait for vLLM and define JSON client.

import requests

def tail_log(path, n=80):
    # Read last log lines.
    path = Path(path)
    if not path.exists():
        return ""
    lines = path.read_text(errors="ignore").splitlines()
    return "\n".join(lines[-n:])

ready = False
MAX_WAIT_SEC = 1800
SLEEP_SEC = 5
PRINT_EVERY_SEC = 60

start = time.perf_counter()
last_print = -PRINT_EVERY_SEC

for step in range(MAX_WAIT_SEC // SLEEP_SEC):
    elapsed = int(time.perf_counter() - start)
    return_code = proc.poll()

    if return_code is not None:
        print(f"vLLM process exited. Return code: {return_code}")
        print("\n=== Last vLLM log lines ===")
        print(tail_log(SERVER_LOG_PATH, n=160))
        raise RuntimeError("vLLM server crashed or exited during startup.")

    try:
        h = requests.get(f"http://localhost:{PORT}/health", timeout=5)
        if h.status_code == 200:
            m = requests.get(f"{BASE_URL}/models", timeout=10)
            if m.status_code == 200:
                ready = True
                model_info = m.json()["data"][0]
                print("vLLM server is ready.")
                print("Model:", model_info["id"])
                print("Max model len:", model_info.get("max_model_len"))
                break
    except Exception:
        pass

    if elapsed - last_print >= PRINT_EVERY_SEC:
        last_print = elapsed
        print(f"Waiting... {elapsed}s")
        recent = tail_log(SERVER_LOG_PATH, n=12)
        if recent.strip():
            print(recent)
        print("-" * 80)

    time.sleep(SLEEP_SEC)

if not ready:
    print("\n=== Last vLLM log lines ===")
    print(tail_log(SERVER_LOG_PATH, n=160))
    raise RuntimeError("vLLM server did not become ready before timeout.")

client = OpenAI(
    api_key="EMPTY",
    base_url=BASE_URL,
    timeout=3600,
)

LLM_SAMPLING_KWARGS = {
    "temperature": 0.6,
    "top_p": 0.95,
    "presence_penalty": 0.0,
}

LLM_EXTRA_BODY = {
    "top_k": 20,
    "min_p": 0.0,
    "repetition_penalty": 1.0,
    "chat_template_kwargs": {
        "enable_thinking": False,
    },
}

DEFAULT_LLM_MAX_TOKENS = 1024

def render_prompt(template, **kwargs):
    # Replace explicit placeholders only.
    rendered = template
    for key, value in kwargs.items():
        rendered = rendered.replace("{" + key + "}", str(value))
    return rendered

def strip_code_fence(text):
    # Remove markdown fences.
    text = (text or "").strip()
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    return text.strip()

def strip_think_blocks(text):
    # Remove think blocks.
    return re.sub(r"<think>.*?</think>", "", text or "", flags=re.DOTALL).strip()

def extract_json_object(text):
    # Extract first JSON object.
    text = strip_code_fence(strip_think_blocks(text))
    start = text.find("{")
    end = text.rfind("}")

    if start == -1 or end == -1 or end <= start:
        raise ValueError(f"No JSON object found. Raw text:\n{text[:2000]}")

    return text[start:end + 1]

def call_llm_json(
    prompt,
    schema,
    max_tokens=DEFAULT_LLM_MAX_TOKENS,
    max_retries=2,
    print_final=False,
):
    # Call non-thinking Qwen and validate JSON.
    last_error = None
    last_text = None

    for attempt in range(max_retries + 1):
        if attempt == 0:
            current_prompt = prompt
        else:
            current_prompt = (
                prompt
                + "\n\nYour previous response was invalid. "
                + "Return only valid JSON matching the requested schema. "
                + "Do not include markdown or explanations.\n\n"
                + f"Validation error:\n{last_error}\n\n"
                + f"Previous response:\n{last_text}"
            )

        response = client.chat.completions.create(
            model=LLM_MODEL_NAME,
            messages=[{"role": "user", "content": current_prompt}],
            max_tokens=max_tokens,
            **LLM_SAMPLING_KWARGS,
            extra_body=LLM_EXTRA_BODY,
        )

        content = response.choices[0].message.content or ""
        last_text = content

        try:
            json_text = extract_json_object(content)
            parsed = json.loads(json_text)
            validate(instance=parsed, schema=schema)

            if print_final:
                print(json.dumps(parsed, ensure_ascii=False, indent=2))

            return {
                "parsed": parsed,
                "raw_content": content,
                "usage": response.usage,
            }

        except Exception as e:
            last_error = repr(e)

    raise RuntimeError(f"LLM JSON call failed. Last error: {last_error}\nLast response:\n{last_text}")

print("LLM JSON client is ready.")

Waiting... 0s
(APIServer pid=6085) INFO 06-20 12:14:30 [api_utils.py:339] 
(APIServer pid=6085) INFO 06-20 12:14:30 [api_utils.py:339]        █     █     █▄   ▄█
(APIServer pid=6085) INFO 06-20 12:14:30 [api_utils.py:339]  ▄▄ ▄█ █     █     █ ▀▄▀ █  version 0.23.1rc1.dev207+gdced29076
(APIServer pid=6085) INFO 06-20 12:14:30 [api_utils.py:339]   █▄█▀ █     █     █     █  model   Qwen/Qwen3.5-27B
(APIServer pid=6085) INFO 06-20 12:14:30 [api_utils.py:339]    ▀▀  ▀▀▀▀▀ ▀▀▀▀▀ ▀     ▀
(APIServer pid=6085) INFO 06-20 12:14:30 [api_utils.py:339] 
(APIServer pid=6085) INFO 06-20 12:14:30 [api_utils.py:273] non-default args: {'model_tag': 'Qwen/Qwen3.5-27B', 'default_chat_template_kwargs': {'enable_thinking': False}, 'host': '0.0.0.0', 'model': 'Qwen/Qwen3.5-27B', 'trust_remote_code': True, 'dtype': 'bfloat16', 'max_model_len': 32768, 'generation_config': 'vllm', 'reasoning_parser': 'qwen3', 'gpu_memory_utilization': 0.7, 'enable_prefix_caching': True, 'language_model_only': True, 'max_num_bat

In [ ]:
# cell 18
# Load Qwen embedding model.

from transformers import AutoTokenizer, AutoModel

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available.")

def print_gpu_mem(prefix):
    # Print torch GPU memory.
    torch.cuda.synchronize()
    allocated = torch.cuda.memory_allocated() / (1024 ** 3)
    reserved = torch.cuda.memory_reserved() / (1024 ** 3)
    print(f"{prefix}: torch allocated={allocated:.2f} GB, reserved={reserved:.2f} GB")

gc.collect()
torch.cuda.empty_cache()

embed_tokenizer = AutoTokenizer.from_pretrained(
    EMBED_MODEL_NAME,
    padding_side="left",
    trust_remote_code=True,
)

def load_embedding_model():
    # Load embedding model with attention fallback.
    try:
        model = AutoModel.from_pretrained(
            EMBED_MODEL_NAME,
            dtype=torch.bfloat16,
            attn_implementation="flash_attention_2",
            trust_remote_code=True,
            low_cpu_mem_usage=True,
        ).cuda().eval()
        print("Loaded embedding model with flash_attention_2.")
        return model
    except Exception as e:
        print("flash_attention_2 failed for embedding. Falling back to sdpa.")
        print("Error:", repr(e))
        gc.collect()
        torch.cuda.empty_cache()

        model = AutoModel.from_pretrained(
            EMBED_MODEL_NAME,
            dtype=torch.bfloat16,
            attn_implementation="sdpa",
            trust_remote_code=True,
            low_cpu_mem_usage=True,
        ).cuda().eval()
        print("Loaded embedding model with sdpa.")
        return model

embed_model = load_embedding_model()

def last_token_pool(last_hidden_states: Tensor, attention_mask: Tensor) -> Tensor:
    # Pool last non-padding token.
    left_padding = attention_mask[:, -1].sum() == attention_mask.shape[0]

    if left_padding:
        return last_hidden_states[:, -1]

    sequence_lengths = attention_mask.sum(dim=1) - 1
    batch_size = last_hidden_states.shape[0]

    return last_hidden_states[
        torch.arange(batch_size, device=last_hidden_states.device),
        sequence_lengths,
    ]

def get_detailed_instruct(task_description, query):
    # Format Qwen embedding query instruction.
    return f"Instruct: {task_description}\nQuery: {query}"

EMBED_QUERY_TASK = (
    "Given a retrieval query, retrieve relevant knowledge-graph relation or fact texts that answer the query"
)

@torch.inference_mode()
def encode_texts(
    texts,
    is_query=False,
    batch_size=16,
):
    # Encode texts with normalized float32 embeddings. No truncation.
    if isinstance(texts, str):
        texts = [texts]

    if is_query:
        input_texts = [
            get_detailed_instruct(EMBED_QUERY_TASK, text)
            for text in texts
        ]
    else:
        input_texts = list(texts)

    all_embeddings = []

    for start in range(0, len(input_texts), batch_size):
        batch_texts = input_texts[start:start + batch_size]

        batch = embed_tokenizer(
            batch_texts,
            padding=True,
            truncation=False,
            return_tensors="pt",
        ).to("cuda")

        outputs = embed_model(**batch)

        emb = last_token_pool(outputs.last_hidden_state, batch["attention_mask"])
        emb = F.normalize(emb.float(), p=2, dim=1)

        all_embeddings.append(emb.cpu().numpy().astype(np.float32))

    return np.concatenate(all_embeddings, axis=0)

print("Embedding model:", EMBED_MODEL_NAME)
print("Embedding truncation:", False)
print_gpu_mem("After embedding load")

config.json:   0%|          | 0.00/729 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.26k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/30.4k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

flash_attention_2 failed for embedding. Falling back to sdpa.
Error: ImportError("FlashAttention2 has been toggled on, but it cannot be used due to the following error: the package for FlashAttention2 doesn't seem to be installed.")


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Loaded embedding model with sdpa.
Embedding model: Qwen/Qwen3-Embedding-8B
Embedding truncation: False
After embedding load: torch allocated=14.10 GB, reserved=14.24 GB


In [ ]:
# cell 19
# Load Qwen reranker model.

from transformers import AutoModelForCausalLM

gc.collect()
torch.cuda.empty_cache()

rerank_tokenizer = AutoTokenizer.from_pretrained(
    RERANKER_MODEL_NAME,
    padding_side="left",
    trust_remote_code=True,
)

if rerank_tokenizer.pad_token is None:
    rerank_tokenizer.pad_token = rerank_tokenizer.eos_token

def load_reranker_model():
    # Load reranker model with attention fallback.
    try:
        model = AutoModelForCausalLM.from_pretrained(
            RERANKER_MODEL_NAME,
            dtype=torch.bfloat16,
            attn_implementation="flash_attention_2",
            trust_remote_code=True,
            low_cpu_mem_usage=True,
        ).cuda().eval()
        print("Loaded reranker with flash_attention_2.")
        return model
    except Exception as e:
        print("flash_attention_2 failed for reranker. Falling back to sdpa.")
        print("Error:", repr(e))
        gc.collect()
        torch.cuda.empty_cache()

        model = AutoModelForCausalLM.from_pretrained(
            RERANKER_MODEL_NAME,
            dtype=torch.bfloat16,
            attn_implementation="sdpa",
            trust_remote_code=True,
            low_cpu_mem_usage=True,
        ).cuda().eval()
        print("Loaded reranker with sdpa.")
        return model

rerank_model = load_reranker_model()

token_true_id = rerank_tokenizer("yes", add_special_tokens=False).input_ids[0]
token_false_id = rerank_tokenizer("no", add_special_tokens=False).input_ids[0]

RERANK_PREFIX = (
    "<|im_start|>system\n"
    "Judge whether the Document meets the requirements based on the Query and the Instruct provided. "
    "Note that the answer can only be \"yes\" or \"no\"."
    "<|im_end|>\n"
    "<|im_start|>user\n"
)

RERANK_SUFFIX = (
    "<|im_end|>\n"
    "<|im_start|>assistant\n"
    "<think>\n\n</think>\n\n"
)

rerank_prefix_tokens = rerank_tokenizer.encode(RERANK_PREFIX, add_special_tokens=False)
rerank_suffix_tokens = rerank_tokenizer.encode(RERANK_SUFFIX, add_special_tokens=False)

RERANK_TASK = "Given a web search query, retrieve relevant passages that answer the query"

def format_rerank_instruction(query, doc, instruction=RERANK_TASK):
    # Format reranker input.
    return f"<Instruct>: {instruction}\n<Query>: {query}\n<Document>: {doc}"

def process_reranker_inputs(pairs):
    # Tokenize reranker pairs. No truncation.
    texts = [
        format_rerank_instruction(query, doc)
        for query, doc in pairs
    ]

    inputs = rerank_tokenizer(
        texts,
        padding=False,
        truncation=False,
        return_attention_mask=False,
    )

    for i, ids in enumerate(inputs["input_ids"]):
        inputs["input_ids"][i] = rerank_prefix_tokens + ids + rerank_suffix_tokens

    inputs = rerank_tokenizer.pad(
        inputs,
        padding=True,
        return_tensors="pt",
    )

    return {k: v.to("cuda") for k, v in inputs.items()}

@torch.inference_mode()
def rerank_pairs(pairs, batch_size=4):
    # Score query-document pairs.
    if not pairs:
        return []

    all_scores = []

    for start in range(0, len(pairs), batch_size):
        batch_pairs = pairs[start:start + batch_size]
        inputs = process_reranker_inputs(batch_pairs)

        logits = rerank_model(**inputs).logits[:, -1, :]
        true_logits = logits[:, token_true_id]
        false_logits = logits[:, token_false_id]

        yes_no_logits = torch.stack([false_logits, true_logits], dim=1)
        scores = F.log_softmax(yes_no_logits.float(), dim=1)[:, 1].exp()

        all_scores.extend(scores.detach().cpu().tolist())

    return all_scores

def rerank_documents(query, documents, doc_ids=None, top_k=None, batch_size=4):
    # Rerank documents for one query.
    if not documents:
        return []

    if doc_ids is None:
        doc_ids = list(range(len(documents)))

    pairs = [(query, doc) for doc in documents]
    scores = rerank_pairs(pairs, batch_size=batch_size)

    order = np.argsort(-np.asarray(scores))

    if top_k is not None:
        order = order[:top_k]

    results = []
    for rank, idx in enumerate(order, start=1):
        results.append({
            "rank": rank,
            "doc_id": doc_ids[idx],
            "score": float(scores[idx]),
            "text": documents[idx],
        })

    return results

print("Reranker model:", RERANKER_MODEL_NAME)
print("Reranker truncation:", False)
print_gpu_mem("After reranker load")

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.71k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

flash_attention_2 failed for reranker. Falling back to sdpa.
Error: ImportError("FlashAttention2 has been toggled on, but it cannot be used due to the following error: the package for FlashAttention2 doesn't seem to be installed.")


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

Loaded reranker with sdpa.
Reranker model: Qwen/Qwen3-Reranker-4B
Reranker truncation: False
After reranker load: torch allocated=21.64 GB, reserved=21.78 GB


In [ ]:
# cell 20
# Retrieval and traversal helpers.

def topk_indices_from_scores(scores, top_k):
    # Return top-k indices sorted by score.
    scores = np.asarray(scores, dtype=np.float32)

    if scores.size == 0:
        return np.array([], dtype=np.int64)

    top_k = min(int(top_k), scores.size)

    if top_k <= 0:
        return np.array([], dtype=np.int64)

    idx = np.argpartition(-scores, top_k - 1)[:top_k]
    idx = idx[np.argsort(-scores[idx])]

    return idx.astype(np.int64)

def score_embeddings(query_embedding, matrix_rows):
    # Dot product over normalized vectors.
    query_embedding = np.asarray(query_embedding, dtype=np.float32).reshape(-1)
    matrix_rows = np.asarray(matrix_rows, dtype=np.float32)
    return matrix_rows @ query_embedding

def search_entity_dense(query_entity, top_k=4):
    # Dense entity FAISS search.
    query_embedding = encode_texts(
        [normalize_entity_text(query_entity)],
        is_query=False,
        batch_size=1,
    )

    scores, ids = entity_faiss_index.search(query_embedding.astype(np.float32), top_k)

    results = []
    for score, entity_id in zip(scores[0], ids[0]):
        if int(entity_id) == -1:
            continue

        results.append({
            "entity_id": int(entity_id),
            "entity": id_to_entity.get(int(entity_id)),
            "score": float(score),
        })

    return results

def retrieve_entity_candidates_for_anchor(
    anchor_entity,
    top_k_token,
    top_k_char3,
    top_k_dense,
):
    # Retrieve KG entity candidates.
    token_hits = search_entity_bm25(
        query=anchor_entity,
        index_payload=token_bm25_index,
        top_k=top_k_token,
    )

    char3_hits = search_entity_bm25(
        query=anchor_entity,
        index_payload=char3_bm25_index,
        top_k=top_k_char3,
    )

    dense_hits = search_entity_dense(
        query_entity=anchor_entity,
        top_k=top_k_dense,
    )

    merged = {}

    for hit in token_hits:
        eid = int(hit["entity_id"])
        merged.setdefault(eid, {
            "entity_id": eid,
            "entity": hit["entity"],
            "token_bm25_score": 0.0,
            "char3_bm25_score": 0.0,
            "dense_score": 0.0,
            "sources": [],
        })
        merged[eid]["token_bm25_score"] = float(hit["score"])
        merged[eid]["sources"].append("token_bm25")

    for hit in char3_hits:
        eid = int(hit["entity_id"])
        merged.setdefault(eid, {
            "entity_id": eid,
            "entity": hit["entity"],
            "token_bm25_score": 0.0,
            "char3_bm25_score": 0.0,
            "dense_score": 0.0,
            "sources": [],
        })
        merged[eid]["char3_bm25_score"] = float(hit["score"])
        merged[eid]["sources"].append("char3_bm25")

    for hit in dense_hits:
        eid = int(hit["entity_id"])
        merged.setdefault(eid, {
            "entity_id": eid,
            "entity": hit["entity"],
            "token_bm25_score": 0.0,
            "char3_bm25_score": 0.0,
            "dense_score": 0.0,
            "sources": [],
        })
        merged[eid]["dense_score"] = float(hit["score"])
        merged[eid]["sources"].append("dense_faiss")

    return list(merged.values())

def get_chunk_record_by_chunk_id(chunk_id):
    # Get chunk text record.
    rec = chunk_text_by_chunk_id.get(str(chunk_id))
    if rec is None:
        raise KeyError(f"Chunk not found: {chunk_id}")
    return rec

def make_chunk_candidate(
    chunk_id,
    retrieval_query,
    source_type,
    source_id,
    score,
    source_payload=None,
):
    # Build chunk candidate.
    chunk_rec = get_chunk_record_by_chunk_id(chunk_id)

    return {
        "candidate_uid": f"{source_type}:{source_id}:{chunk_id}",
        "chunk_id": str(chunk_id),
        "chunk_node_id": int(chunk_rec["chunk_node_id"]),
        "title": chunk_rec["title"],
        "text": chunk_rec["text"],
        "retrieval_query": retrieval_query,
        "source_type": source_type,
        "source_id": str(source_id),
        "score": float(score),
        "source_payload": source_payload or {},
    }

def make_relation_path_candidate(
    path_id,
    retrieval_query,
    source_entity_id,
    source_entity,
    terminal_entity_id,
    terminal_entity,
    relation_id,
    relation_text,
    chunk_id,
    relation_score,
    readable_path,
    parent_path_id=None,
    parent_readable_path=None,
    path_depth=1,
    source_type="relation",
):
    # Build relation path candidate.
    return {
        "path_id": str(path_id),
        "path_depth": int(path_depth),
        "parent_path_id": parent_path_id,
        "retrieval_query": retrieval_query,
        "source_entity_id": int(source_entity_id),
        "source_entity": source_entity,
        "terminal_entity_id": int(terminal_entity_id),
        "terminal_entity": terminal_entity,
        "relation_id": int(relation_id),
        "relation": relation_text,
        "chunk_id": str(chunk_id),
        "relation_score": float(relation_score),
        "readable_path": readable_path,
        "parent_readable_path": parent_readable_path,
        "source_type": source_type,
    }

def retrieve_top_relations_from_entity(
    entity_id,
    retrieval_query,
    query_embedding,
    top_k=3,
):
    # Retrieve top outgoing relation edges.
    rel_positions = relation_out_positions_by_entity.get(int(entity_id), [])

    if not rel_positions:
        return []

    rel_positions_np = np.asarray(rel_positions, dtype=np.int64)
    rel_relation_ids = rel_ids_np[rel_positions_np]
    rel_vecs = relation_embeddings[rel_relation_ids]

    rel_scores = score_embeddings(query_embedding, rel_vecs)
    rel_top_local = topk_indices_from_scores(rel_scores, top_k)

    results = []

    for local_rank, local_idx in enumerate(rel_top_local, start=1):
        pos = int(rel_positions_np[int(local_idx)])
        relation_id = int(rel_ids_np[pos])
        score = float(rel_scores[int(local_idx)])

        edge_rec = relation_record_by_id[relation_id]

        src_entity_id = int(rel_src_np[pos])
        dst_entity_id = int(rel_dst_np[pos])

        src_entity = id_to_entity.get(src_entity_id)
        dst_entity = id_to_entity.get(dst_entity_id)

        chunk_node_id = int(rel_chunk_node_ids_np[pos])
        chunk_id = str(chunk_node_id_to_chunk_id[str(chunk_node_id)])

        results.append({
            "rank": local_rank,
            "relation_id": relation_id,
            "score": score,
            "source_entity_id": src_entity_id,
            "source_entity": src_entity,
            "terminal_entity_id": dst_entity_id,
            "terminal_entity": dst_entity,
            "relation": edge_rec["relation"],
            "chunk_node_id": chunk_node_id,
            "chunk_id": chunk_id,
        })

    return results

def retrieve_top_facts_from_entity(
    entity_id,
    retrieval_query,
    query_embedding,
    top_k=3,
):
    # Retrieve top fact edges.
    fact_positions = fact_positions_by_entity.get(int(entity_id), [])

    if not fact_positions:
        return []

    fact_positions_np = np.asarray(fact_positions, dtype=np.int64)
    local_fact_ids = fact_ids_np[fact_positions_np]
    fact_vecs = fact_embeddings[local_fact_ids]

    fact_scores = score_embeddings(query_embedding, fact_vecs)
    fact_top_local = topk_indices_from_scores(fact_scores, top_k)

    results = []

    for local_rank, local_idx in enumerate(fact_top_local, start=1):
        pos = int(fact_positions_np[int(local_idx)])
        fact_id = int(fact_ids_np[pos])
        score = float(fact_scores[int(local_idx)])

        edge_rec = fact_record_by_id[fact_id]

        chunk_node_id = int(fact_dst_chunk_np[pos])
        chunk_id = str(chunk_node_id_to_chunk_id[str(chunk_node_id)])

        results.append({
            "rank": local_rank,
            "fact_id": fact_id,
            "score": score,
            "entity_id": int(entity_id),
            "entity": id_to_entity.get(int(entity_id)),
            "info": edge_rec["info"],
            "chunk_node_id": chunk_node_id,
            "chunk_id": chunk_id,
        })

    return results

def rerank_chunk_candidates_for_query(candidate_items, query, top_k, batch_size=2):
    # Rerank chunk candidates for one query.
    if not candidate_items:
        return []

    documents = [
        f"Title: {item['title']}\n\n{item['text']}"
        for item in candidate_items
    ]

    doc_ids = [item["candidate_uid"] for item in candidate_items]
    item_by_uid = {item["candidate_uid"]: item for item in candidate_items}

    reranked = rerank_documents(
        query=query,
        documents=documents,
        doc_ids=doc_ids,
        top_k=None,
        batch_size=batch_size,
    )

    scored_items = []
    for r in reranked:
        uid = r["doc_id"]
        item = dict(item_by_uid[uid])
        item["reranker_score"] = float(r["score"])
        scored_items.append(item)

    best_by_chunk_id = {}
    for item in scored_items:
        chunk_id = item["chunk_id"]
        if chunk_id not in best_by_chunk_id:
            best_by_chunk_id[chunk_id] = item
        elif item["reranker_score"] > best_by_chunk_id[chunk_id]["reranker_score"]:
            best_by_chunk_id[chunk_id] = item

    unique_items = list(best_by_chunk_id.values())
    unique_items.sort(key=lambda x: x["reranker_score"], reverse=True)

    return unique_items[:top_k]

def make_prompt_path_items(path_items, compact_fn, prefix="p"):
    # Build prompt-local path IDs.
    prompt_items = []
    prompt_path_id_to_real_path_id = {}
    real_path_id_to_prompt_path_id = {}

    for i, item in enumerate(path_items, start=1):
        prompt_path_id = f"{prefix}{i}"
        compact_item = compact_fn(item)
        real_path_id = compact_item["path_id"]

        prompt_path_id_to_real_path_id[prompt_path_id] = real_path_id
        real_path_id_to_prompt_path_id[real_path_id] = prompt_path_id

        compact_item = dict(compact_item)
        compact_item["path_id"] = prompt_path_id
        prompt_items.append(compact_item)

    return prompt_items, prompt_path_id_to_real_path_id, real_path_id_to_prompt_path_id

def restore_selected_path_queries_from_prompt(selected_path_queries, prompt_path_id_to_real_path_id):
    # Restore prompt-local path IDs.
    restored = []
    invalid_prompt_path_ids = []

    for item in selected_path_queries:
        prompt_path_id = item.get("path_id", "")

        if prompt_path_id not in prompt_path_id_to_real_path_id:
            invalid_prompt_path_ids.append(prompt_path_id)
            continue

        restored_item = dict(item)
        restored_item["prompt_path_id"] = prompt_path_id
        restored_item["path_id"] = prompt_path_id_to_real_path_id[prompt_path_id]
        restored.append(restored_item)

    return restored, invalid_prompt_path_ids

def restore_selected_path_ids_from_prompt(selected_path_ids, prompt_path_id_to_real_path_id):
    # Restore final selected path IDs.
    restored = []
    invalid_prompt_path_ids = []

    for prompt_path_id in selected_path_ids:
        if prompt_path_id not in prompt_path_id_to_real_path_id:
            invalid_prompt_path_ids.append(prompt_path_id)
            continue
        restored.append(prompt_path_id_to_real_path_id[prompt_path_id])

    return restored, invalid_prompt_path_ids

print("Retrieval and traversal helpers are ready.")

Retrieval and traversal helpers are ready.


In [ ]:
# cell 21
# Final readiness check.

for name in REQUIRED_PROMPT_OBJECTS:
    if name not in globals():
        raise RuntimeError(f"Missing required prompt/schema object: {name}")

required_runtime_objects = [
    "question_records",
    "kg",
    "kg_meta",
    "id_to_entity",
    "entity_to_id",
    "chunk_text_by_chunk_id",
    "relation_record_by_id",
    "fact_record_by_id",
    "token_bm25_index",
    "char3_bm25_index",
    "entity_faiss_index",
    "relation_embeddings",
    "fact_embeddings",
    "relation_out_positions_by_entity",
    "fact_positions_by_entity",
    "client",
    "embed_model",
    "rerank_model",
]

for name in required_runtime_objects:
    if name not in globals():
        raise RuntimeError(f"Missing runtime object: {name}")

print("=" * 100)
print("PHASE 1 READY")
print("=" * 100)
print("Dataset:", DATASET)
print("Questions:", len(question_records))
print("Evidence output:", EVIDENCE_OUTPUT_PATH)
print("KG node types:", kg.node_types)
print("KG edge types:", kg.edge_types)
print("Entities:", len(id_to_entity))
print("Chunks:", len(chunk_text_by_chunk_id))
print("Relations:", len(relation_record_by_id))
print("Facts:", len(fact_record_by_id))
print("Relation embeddings:", relation_embeddings.shape)
print("Fact embeddings:", fact_embeddings.shape)
print("LLM server:", BASE_URL)
print("LLM model:", LLM_MODEL_NAME)
print("LLM max model len:", MAX_MODEL_LEN)
print("Embedding model:", EMBED_MODEL_NAME)
print("Reranker model:", RERANKER_MODEL_NAME)
print("Manual truncation for LLM/chunks/reranker/embedding:", False)

ram = psutil.virtual_memory()
print(f"RAM used: {ram.used / (1024 ** 3):.2f} GB / {ram.total / (1024 ** 3):.2f} GB")

print_gpu_mem("Final GPU memory")

!nvidia-smi

PHASE 1 READY
Dataset: 2wikimultihopqa
Questions: 1000
Evidence output: /content/drive/MyDrive/final_project/ablation/3llm_call_(hop)/2wikimultihopqa_dev_2020wiki_1000_traversal_evidence.json
KG node types: ['entity', 'chunk']
KG edge types: [('entity', 'relation', 'entity'), ('entity', 'fact', 'chunk'), ('chunk', 'next_chunk', 'chunk'), ('chunk', 'prev_chunk', 'chunk')]
Entities: 146864
Chunks: 12685
Relations: 247252
Facts: 184001
Relation embeddings: (247252, 4096)
Fact embeddings: (184001, 4096)
LLM server: http://localhost:8000/v1
LLM model: Qwen/Qwen3.5-27B
LLM max model len: 32768
Embedding model: Qwen/Qwen3-Embedding-8B
Reranker model: Qwen/Qwen3-Reranker-4B
Manual truncation for LLM/chunks/reranker/embedding: False
RAM used: 20.18 GB / 176.88 GB
Final GPU memory: torch allocated=21.64 GB, reserved=21.78 GB
Sat Jun 20 12:22:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Versio

In [ ]:
# cell 22
# Phase 2 config and output I/O.

import traceback
from copy import deepcopy

PHASE2_OUTPUT_PATH = EVIDENCE_OUTPUT_PATH
PHASE2_TMP_PATH = EVIDENCE_TMP_PATH

TRAVERSAL_START_INDEX = 0
TRAVERSAL_END_INDEX = None  # None means all questions.

SAVE_EVERY_N_QUESTIONS = 1
CLEAR_CACHE_EVERY_N_QUESTIONS = 25

RERANK_BATCH_SIZE = 2

HOP1_LLM_MAX_TOKENS = 768
HOP23_LLM_MAX_TOKENS = 1024
HOP4_LLM_MAX_TOKENS = 768

def atomic_write_json(path, data):
    # Atomically write JSON to Drive.
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(
        json.dumps(data, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    os.replace(tmp, path)

def load_existing_results(path):
    # Load previous results for resume.
    path = Path(path)
    if not path.exists():
        return {}

    try:
        data = json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return {}

    if not isinstance(data, list):
        return {}

    result_map = {}
    for pos, rec in enumerate(data):
        if not isinstance(rec, dict):
            continue
        source_index = rec.get("source_index", pos)
        try:
            source_index = int(source_index)
        except Exception:
            source_index = pos
        result_map[source_index] = rec

    return result_map

def save_result_map(path, result_map):
    # Save results sorted by original source index.
    ordered = [
        result_map[idx]
        for idx in sorted(result_map.keys())
    ]
    atomic_write_json(path, ordered)

def safe_json_dumps(obj):
    # Stable JSON for prompts.
    return json.dumps(obj, ensure_ascii=False, indent=2)

print("Phase 2 output path:", PHASE2_OUTPUT_PATH)

Phase 2 output path: /content/drive/MyDrive/final_project/ablation/3llm_call_(hop)/2wikimultihopqa_dev_2020wiki_1000_traversal_evidence.json


In [ ]:
# cell 23
# Prompt formatting and selection helpers.

def chunk_item_from_chunk_id(chunk_id):
    # Build a full chunk item from chunk_id.
    rec = get_chunk_record_by_chunk_id(chunk_id)
    return {
        "chunk_id": str(rec["chunk_id"]),
        "chunk_node_id": int(rec["chunk_node_id"]),
        "title": rec["title"],
        "text": rec["text"],
    }

def normalize_chunk_item(item):
    # Normalize any chunk-like item.
    if item is None:
        return None

    if "chunk_id" not in item:
        return None

    chunk_id = str(item["chunk_id"])
    rec = get_chunk_record_by_chunk_id(chunk_id)

    return {
        "chunk_id": chunk_id,
        "chunk_node_id": int(item.get("chunk_node_id", rec["chunk_node_id"])),
        "title": item.get("title", rec["title"]),
        "text": item.get("text", rec["text"]),
        **{
            k: v
            for k, v in item.items()
            if k not in {"chunk_id", "chunk_node_id", "title", "text"}
        },
    }

def compact_chunk_for_prompt(item):
    # Full chunk for LLM prompt. No truncation.
    item = normalize_chunk_item(item)
    return {
        "chunk_id": item["chunk_id"],
        "title": item["title"],
        "text": item["text"],
    }

def compact_path_for_hop23_prompt(item):
    # Full path metadata for HOP2/HOP3 prompt.
    return {
        "path_id": item["path_id"],
        "path": item["readable_path"],
        "terminal_entity": item["terminal_entity"],
        "source_chunk_id": item["chunk_id"],
        "path_depth": int(item.get("path_depth", 1)),
    }

def compact_path_for_hop4_prompt(item):
    # Full path metadata for HOP4 prompt.
    return {
        "path_id": item["path_id"],
        "path": item["readable_path"],
        "terminal_entity": item["terminal_entity"],
        "source_chunk_id": item["chunk_id"],
        "path_depth": int(item.get("path_depth", 1)),
    }

def ordered_unique(values):
    # Preserve order and remove duplicates.
    seen = set()
    out = []
    for value in values:
        value = str(value)
        if value in seen:
            continue
        seen.add(value)
        out.append(value)
    return out

def dedupe_chunks_keep_best(items, score_key="reranker_score"):
    # Deduplicate chunks by chunk_id.
    best = {}

    for item in items:
        item = normalize_chunk_item(item)
        if item is None:
            continue

        chunk_id = item["chunk_id"]
        score = float(item.get(score_key, item.get("score", 0.0)) or 0.0)

        if chunk_id not in best:
            best[chunk_id] = item
        else:
            old_score = float(best[chunk_id].get(score_key, best[chunk_id].get("score", 0.0)) or 0.0)
            if score > old_score:
                best[chunk_id] = item

    return list(best.values())

def dedupe_paths_by_id(items):
    # Deduplicate paths by internal path_id.
    out = {}
    for item in items:
        if not item or "path_id" not in item:
            continue
        out[str(item["path_id"])] = item
    return list(out.values())

def get_required_count(max_count, available_count):
    # Match prompt quota rule.
    return min(int(max_count), int(available_count))

def sanitize_selected_chunk_ids(selected_ids, available_chunk_ids, max_count=6):
    # Keep valid IDs and fill from available IDs if needed.
    available_chunk_ids = ordered_unique(available_chunk_ids)
    available_set = set(available_chunk_ids)

    clean = []
    for chunk_id in selected_ids or []:
        chunk_id = str(chunk_id)
        if chunk_id in available_set and chunk_id not in clean:
            clean.append(chunk_id)

    required = get_required_count(max_count, len(available_chunk_ids))

    if len(clean) < required:
        for chunk_id in available_chunk_ids:
            if chunk_id not in clean:
                clean.append(chunk_id)
            if len(clean) >= required:
                break

    return clean[:required]

def sanitize_selected_path_queries(selected_path_queries, prompt_paths, prompt_to_real, max_count=5):
    # Keep valid selected path queries and fill missing paths if needed.
    prompt_path_by_id = {
        item["path_id"]: item
        for item in prompt_paths
    }

    clean = []
    used_prompt_ids = set()

    for item in selected_path_queries or []:
        prompt_path_id = str(item.get("path_id", ""))
        if prompt_path_id not in prompt_to_real:
            continue
        if prompt_path_id in used_prompt_ids:
            continue

        prompt_path = prompt_path_by_id[prompt_path_id]
        clean.append({
            "prompt_path_id": prompt_path_id,
            "path_id": prompt_to_real[prompt_path_id],
            "terminal_entity": item.get("terminal_entity", prompt_path["terminal_entity"]),
            "query": item.get(
                "query",
                f"What information about {prompt_path['terminal_entity']} is needed to answer the original question?"
            ),
        })
        used_prompt_ids.add(prompt_path_id)

    required = get_required_count(max_count, len(prompt_paths))

    if len(clean) < required:
        for prompt_path in prompt_paths:
            prompt_path_id = prompt_path["path_id"]
            if prompt_path_id in used_prompt_ids:
                continue

            clean.append({
                "prompt_path_id": prompt_path_id,
                "path_id": prompt_to_real[prompt_path_id],
                "terminal_entity": prompt_path["terminal_entity"],
                "query": f"What information about {prompt_path['terminal_entity']} is needed to answer the original question?",
            })
            used_prompt_ids.add(prompt_path_id)

            if len(clean) >= required:
                break

    return clean[:required]

def sanitize_selected_path_ids(selected_path_ids, prompt_paths, prompt_to_real, max_count=5):
    # Keep valid final selected path IDs and fill missing paths if needed.
    prompt_path_ids = [item["path_id"] for item in prompt_paths]
    prompt_set = set(prompt_path_ids)

    clean_prompt_ids = []
    for prompt_path_id in selected_path_ids or []:
        prompt_path_id = str(prompt_path_id)
        if prompt_path_id in prompt_set and prompt_path_id not in clean_prompt_ids:
            clean_prompt_ids.append(prompt_path_id)

    required = get_required_count(max_count, len(prompt_path_ids))

    if len(clean_prompt_ids) < required:
        for prompt_path_id in prompt_path_ids:
            if prompt_path_id not in clean_prompt_ids:
                clean_prompt_ids.append(prompt_path_id)
            if len(clean_prompt_ids) >= required:
                break

    return [
        prompt_to_real[prompt_path_id]
        for prompt_path_id in clean_prompt_ids[:required]
    ]

def add_path_lineage(path_item, parent_path=None):
    # Add ordered chunk IDs along a path.
    path_item = dict(path_item)

    if parent_path is None:
        parent_chunk_ids = []
    else:
        parent_chunk_ids = list(parent_path.get("path_chunk_ids", []))
        if not parent_chunk_ids and parent_path.get("chunk_id"):
            parent_chunk_ids = [str(parent_path["chunk_id"])]

    current_chunk_id = str(path_item["chunk_id"])
    path_item["path_chunk_ids"] = ordered_unique(parent_chunk_ids + [current_chunk_id])
    return path_item

print("Phase 2 prompt helpers are ready.")

Phase 2 prompt helpers are ready.


In [ ]:
# cell 24
# HOP1 retrieval and reranking.

def get_hop1_retrieval_settings(num_anchor_entities):
    # Select retrieval quota based on anchor count.
    if num_anchor_entities == 2:
        return 4, 4
    if num_anchor_entities == 3:
        return 3, 3
    if num_anchor_entities == 4:
        return 2, 2
    raise RuntimeError(f"Unsupported HOP1 entity count: {num_anchor_entities}")

def run_hop1_retrieval(question):
    # Run HOP1 LLM and retrieve initial chunks/paths.
    hop1_prompt = render_prompt(
        HOP1_PROMPT,
        question=question,
    )

    hop1_result = call_llm_json(
        prompt=hop1_prompt,
        schema=HOP1_SCHEMA,
        max_tokens=HOP1_LLM_MAX_TOKENS,
        max_retries=2,
        print_final=False,
    )

    entity_queries = hop1_result["parsed"]["entity_queries"]
    num_anchor_entities = len(entity_queries)

    top_k_per_method, final_quota_per_group = get_hop1_retrieval_settings(num_anchor_entities)

    entity_candidate_groups = []

    for group_id, item in enumerate(entity_queries):
        anchor_entity = item["entity"]
        query = item["query"]

        candidates = retrieve_entity_candidates_for_anchor(
            anchor_entity=anchor_entity,
            top_k_token=top_k_per_method,
            top_k_char3=top_k_per_method,
            top_k_dense=top_k_per_method,
        )

        entity_candidate_groups.append({
            "group_id": group_id,
            "anchor_entity": anchor_entity,
            "query": query,
            "candidates": candidates,
        })

    relation_path_candidates = []
    chunk_candidates_by_group = defaultdict(list)
    path_candidates_by_group = defaultdict(list)

    for group in entity_candidate_groups:
        group_id = int(group["group_id"])
        group_query = group["query"]

        query_embedding = encode_texts(
            [group_query],
            is_query=True,
            batch_size=1,
        )[0]

        for cand in group["candidates"]:
            entity_id = int(cand["entity_id"])
            entity_text = cand["entity"]

            relation_hits = retrieve_top_relations_from_entity(
                entity_id=entity_id,
                retrieval_query=group_query,
                query_embedding=query_embedding,
                top_k=TRAVERSAL_CONFIG["hop1_relation_top_k_per_entity"],
            )

            for hit in relation_hits:
                path_id = (
                    f"hop1_g{group_id}_entity{entity_id}_"
                    f"rel{hit['relation_id']}_rank{hit['rank']}"
                )

                readable_path = (
                    f"{hit['source_entity']} --[{hit['relation']}]--> {hit['terminal_entity']}"
                )

                path_item = make_relation_path_candidate(
                    path_id=path_id,
                    retrieval_query=group_query,
                    source_entity_id=hit["source_entity_id"],
                    source_entity=hit["source_entity"],
                    terminal_entity_id=hit["terminal_entity_id"],
                    terminal_entity=hit["terminal_entity"],
                    relation_id=hit["relation_id"],
                    relation_text=hit["relation"],
                    chunk_id=hit["chunk_id"],
                    relation_score=hit["score"],
                    readable_path=readable_path,
                    parent_path_id=None,
                    parent_readable_path=None,
                    path_depth=1,
                    source_type="hop1_relation",
                )

                path_item.update({
                    "group_id": group_id,
                    "anchor_entity": group["anchor_entity"],
                    "group_query": group_query,
                    "matched_entity_id": entity_id,
                    "matched_entity": entity_text,
                })

                path_item = add_path_lineage(path_item)
                relation_path_candidates.append(path_item)
                path_candidates_by_group[group_id].append(path_item)

                chunk_item = make_chunk_candidate(
                    chunk_id=hit["chunk_id"],
                    retrieval_query=group_query,
                    source_type="hop1_relation",
                    source_id=path_id,
                    score=hit["score"],
                    source_payload={
                        "group_id": group_id,
                        "relation_id": hit["relation_id"],
                        "terminal_entity": hit["terminal_entity"],
                    },
                )
                chunk_item["group_id"] = group_id
                chunk_candidates_by_group[group_id].append(chunk_item)

            fact_hits = retrieve_top_facts_from_entity(
                entity_id=entity_id,
                retrieval_query=group_query,
                query_embedding=query_embedding,
                top_k=TRAVERSAL_CONFIG["hop1_fact_top_k_per_entity"],
            )

            for hit in fact_hits:
                source_id = (
                    f"hop1_g{group_id}_entity{entity_id}_"
                    f"fact{hit['fact_id']}_rank{hit['rank']}"
                )

                chunk_item = make_chunk_candidate(
                    chunk_id=hit["chunk_id"],
                    retrieval_query=group_query,
                    source_type="hop1_fact",
                    source_id=source_id,
                    score=hit["score"],
                    source_payload={
                        "group_id": group_id,
                        "fact_id": hit["fact_id"],
                        "entity": hit["entity"],
                    },
                )
                chunk_item["group_id"] = group_id
                chunk_candidates_by_group[group_id].append(chunk_item)

    selected_hop1_chunks = []

    for group in entity_candidate_groups:
        group_id = int(group["group_id"])
        group_query = group["query"]
        chunk_items = chunk_candidates_by_group[group_id]

        reranked_chunks = rerank_chunk_candidates_for_query(
            candidate_items=chunk_items,
            query=group_query,
            top_k=final_quota_per_group,
            batch_size=RERANK_BATCH_SIZE,
        )

        for item in reranked_chunks:
            item = dict(item)
            item["group_id"] = group_id
            item["anchor_entity"] = group["anchor_entity"]
            item["group_query"] = group_query
            selected_hop1_chunks.append(item)

    selected_hop1_paths = []

    for group in entity_candidate_groups:
        group_id = int(group["group_id"])
        group_query = group["query"]
        path_items = path_candidates_by_group[group_id]

        if not path_items:
            continue

        documents = [
            f"Path: {item['readable_path']}\nSource chunk id: {item['chunk_id']}"
            for item in path_items
        ]

        doc_ids = [
            item["path_id"]
            for item in path_items
        ]

        reranked_paths = rerank_documents(
            query=group_query,
            documents=documents,
            doc_ids=doc_ids,
            top_k=final_quota_per_group,
            batch_size=RERANK_BATCH_SIZE,
        )

        path_by_id = {
            item["path_id"]: item
            for item in path_items
        }

        for r in reranked_paths:
            path_id = r["doc_id"]
            item = dict(path_by_id[path_id])
            item["reranker_score"] = float(r["score"])
            selected_hop1_paths.append(item)

    selected_hop1_chunks = dedupe_chunks_keep_best(selected_hop1_chunks)
    selected_hop1_paths = dedupe_paths_by_id(selected_hop1_paths)

    path_lookup = {
        item["path_id"]: item
        for item in selected_hop1_paths
    }

    return {
        "hop1_entity_queries": entity_queries,
        "selected_chunks": selected_hop1_chunks,
        "selected_paths": selected_hop1_paths,
        "path_lookup": path_lookup,
    }

print("HOP1 retrieval function is ready.")

HOP1 retrieval function is ready.


In [ ]:
# cell 25
# Path and new-entity expansion.

def rerank_path_expansion_chunks(chunk_candidates):
    # Select top chunks from selected-path expansion.
    if not chunk_candidates:
        return []

    chunks_by_query = defaultdict(list)

    for item in chunk_candidates:
        chunks_by_query[item["retrieval_query"]].append(item)

    scored_pool = []

    for query, items in chunks_by_query.items():
        reranked_items = rerank_chunk_candidates_for_query(
            candidate_items=items,
            query=query,
            top_k=len(items),
            batch_size=RERANK_BATCH_SIZE,
        )
        scored_pool.extend(reranked_items)

    selected = dedupe_chunks_keep_best(scored_pool)
    selected.sort(key=lambda x: float(x.get("reranker_score", 0.0)), reverse=True)

    return selected[:TRAVERSAL_CONFIG["path_expansion_final_chunk_top_k"]]

def rerank_new_entity_chunks(chunk_candidates):
    # Select top chunks per new-entity query group.
    if not chunk_candidates:
        return []

    by_group = defaultdict(list)

    for item in chunk_candidates:
        by_group[int(item.get("new_entity_group_id", 0))].append(item)

    selected = []

    for group_id, items in by_group.items():
        query = items[0]["retrieval_query"]

        reranked = rerank_chunk_candidates_for_query(
            candidate_items=items,
            query=query,
            top_k=TRAVERSAL_CONFIG["new_entity_final_chunk_top_k_per_group"],
            batch_size=RERANK_BATCH_SIZE,
        )

        for item in reranked:
            item = dict(item)
            item["new_entity_group_id"] = group_id
            selected.append(item)

    selected = dedupe_chunks_keep_best(selected)
    selected.sort(key=lambda x: float(x.get("reranker_score", 0.0)), reverse=True)

    return selected

def expand_selected_paths(hop_label, selected_path_queries, parent_path_lookup):
    # Expand selected paths from their terminal entities.
    relation_candidates = []
    chunk_candidates = []

    for selected_idx, item in enumerate(selected_path_queries):
        parent_path_id = item["path_id"]
        retrieval_query = item["query"]

        parent_path = parent_path_lookup.get(parent_path_id)
        if parent_path is None:
            continue

        terminal_entity_id = int(parent_path["terminal_entity_id"])

        query_embedding = encode_texts(
            [retrieval_query],
            is_query=True,
            batch_size=1,
        )[0]

        relation_hits = retrieve_top_relations_from_entity(
            entity_id=terminal_entity_id,
            retrieval_query=retrieval_query,
            query_embedding=query_embedding,
            top_k=TRAVERSAL_CONFIG["path_relation_top_k"],
        )

        for hit in relation_hits:
            new_path_id = (
                f"{hop_label}_p{selected_idx}_from_{parent_path_id}_"
                f"rel{hit['relation_id']}_rank{hit['rank']}"
            )

            readable_path = (
                f"{parent_path['readable_path']} || "
                f"{hit['source_entity']} --[{hit['relation']}]--> {hit['terminal_entity']}"
            )

            path_item = make_relation_path_candidate(
                path_id=new_path_id,
                retrieval_query=retrieval_query,
                source_entity_id=hit["source_entity_id"],
                source_entity=hit["source_entity"],
                terminal_entity_id=hit["terminal_entity_id"],
                terminal_entity=hit["terminal_entity"],
                relation_id=hit["relation_id"],
                relation_text=hit["relation"],
                chunk_id=hit["chunk_id"],
                relation_score=hit["score"],
                readable_path=readable_path,
                parent_path_id=parent_path_id,
                parent_readable_path=parent_path["readable_path"],
                path_depth=int(parent_path.get("path_depth", 1)) + 1,
                source_type=f"{hop_label}_selected_path_relation",
            )

            path_item = add_path_lineage(path_item, parent_path=parent_path)
            relation_candidates.append(path_item)

            chunk_item = make_chunk_candidate(
                chunk_id=hit["chunk_id"],
                retrieval_query=retrieval_query,
                source_type=f"{hop_label}_selected_path_relation",
                source_id=new_path_id,
                score=hit["score"],
                source_payload={
                    "parent_path_id": parent_path_id,
                    "relation_id": hit["relation_id"],
                    "terminal_entity": hit["terminal_entity"],
                },
            )
            chunk_candidates.append(chunk_item)

        fact_hits = retrieve_top_facts_from_entity(
            entity_id=terminal_entity_id,
            retrieval_query=retrieval_query,
            query_embedding=query_embedding,
            top_k=TRAVERSAL_CONFIG["path_fact_top_k"],
        )

        for hit in fact_hits:
            source_id = (
                f"{hop_label}_p{selected_idx}_from_{parent_path_id}_"
                f"fact{hit['fact_id']}_rank{hit['rank']}"
            )

            chunk_item = make_chunk_candidate(
                chunk_id=hit["chunk_id"],
                retrieval_query=retrieval_query,
                source_type=f"{hop_label}_selected_path_fact",
                source_id=source_id,
                score=hit["score"],
                source_payload={
                    "parent_path_id": parent_path_id,
                    "fact_id": hit["fact_id"],
                    "terminal_entity": parent_path["terminal_entity"],
                },
            )
            chunk_candidates.append(chunk_item)

    return relation_candidates, chunk_candidates

def expand_new_entities(hop_label, new_entity_queries):
    # Retrieve and expand new entities.
    relation_candidates = []
    chunk_candidates = []

    for group_id, item in enumerate(new_entity_queries[:2]):
        entity = item.get("entity", "")
        retrieval_query = item.get("query", "")

        if not entity or not retrieval_query:
            continue

        candidates = retrieve_entity_candidates_for_anchor(
            anchor_entity=entity,
            top_k_token=TRAVERSAL_CONFIG["new_entity_top_k_per_method"],
            top_k_char3=TRAVERSAL_CONFIG["new_entity_top_k_per_method"],
            top_k_dense=TRAVERSAL_CONFIG["new_entity_top_k_per_method"],
        )

        query_embedding = encode_texts(
            [retrieval_query],
            is_query=True,
            batch_size=1,
        )[0]

        for cand in candidates:
            entity_id = int(cand["entity_id"])

            relation_hits = retrieve_top_relations_from_entity(
                entity_id=entity_id,
                retrieval_query=retrieval_query,
                query_embedding=query_embedding,
                top_k=TRAVERSAL_CONFIG["new_entity_relation_top_k"],
            )

            for hit in relation_hits:
                path_id = (
                    f"{hop_label}_new{group_id}_entity{entity_id}_"
                    f"rel{hit['relation_id']}_rank{hit['rank']}"
                )

                readable_path = (
                    f"{hit['source_entity']} --[{hit['relation']}]--> {hit['terminal_entity']}"
                )

                path_item = make_relation_path_candidate(
                    path_id=path_id,
                    retrieval_query=retrieval_query,
                    source_entity_id=hit["source_entity_id"],
                    source_entity=hit["source_entity"],
                    terminal_entity_id=hit["terminal_entity_id"],
                    terminal_entity=hit["terminal_entity"],
                    relation_id=hit["relation_id"],
                    relation_text=hit["relation"],
                    chunk_id=hit["chunk_id"],
                    relation_score=hit["score"],
                    readable_path=readable_path,
                    parent_path_id=None,
                    parent_readable_path=None,
                    path_depth=1,
                    source_type=f"{hop_label}_new_entity_relation",
                )

                path_item.update({
                    "new_entity_group_id": group_id,
                    "new_entity": entity,
                })

                path_item = add_path_lineage(path_item)
                relation_candidates.append(path_item)

                chunk_item = make_chunk_candidate(
                    chunk_id=hit["chunk_id"],
                    retrieval_query=retrieval_query,
                    source_type=f"{hop_label}_new_entity_relation",
                    source_id=path_id,
                    score=hit["score"],
                    source_payload={
                        "new_entity_group_id": group_id,
                        "new_entity": entity,
                        "relation_id": hit["relation_id"],
                        "terminal_entity": hit["terminal_entity"],
                    },
                )
                chunk_item["new_entity_group_id"] = group_id
                chunk_candidates.append(chunk_item)

            fact_hits = retrieve_top_facts_from_entity(
                entity_id=entity_id,
                retrieval_query=retrieval_query,
                query_embedding=query_embedding,
                top_k=TRAVERSAL_CONFIG["new_entity_fact_top_k"],
            )

            for hit in fact_hits:
                source_id = (
                    f"{hop_label}_new{group_id}_entity{entity_id}_"
                    f"fact{hit['fact_id']}_rank{hit['rank']}"
                )

                chunk_item = make_chunk_candidate(
                    chunk_id=hit["chunk_id"],
                    retrieval_query=retrieval_query,
                    source_type=f"{hop_label}_new_entity_fact",
                    source_id=source_id,
                    score=hit["score"],
                    source_payload={
                        "new_entity_group_id": group_id,
                        "new_entity": entity,
                        "fact_id": hit["fact_id"],
                    },
                )
                chunk_item["new_entity_group_id"] = group_id
                chunk_candidates.append(chunk_item)

    return relation_candidates, chunk_candidates

def expand_after_hop(hop_label, selected_path_queries, parent_path_lookup, new_entity_queries):
    # Expand selected paths and new entities for the next hop.
    path_relation_candidates, path_chunk_candidates = expand_selected_paths(
        hop_label=hop_label,
        selected_path_queries=selected_path_queries,
        parent_path_lookup=parent_path_lookup,
    )

    new_entity_relation_candidates, new_entity_chunk_candidates = expand_new_entities(
        hop_label=hop_label,
        new_entity_queries=new_entity_queries,
    )

    selected_path_chunks = rerank_path_expansion_chunks(path_chunk_candidates)
    selected_new_entity_chunks = rerank_new_entity_chunks(new_entity_chunk_candidates)

    candidate_chunks = dedupe_chunks_keep_best(
        selected_path_chunks + selected_new_entity_chunks
    )

    candidate_paths = dedupe_paths_by_id(
        path_relation_candidates + new_entity_relation_candidates
    )

    path_lookup = {
        item["path_id"]: item
        for item in candidate_paths
    }

    return {
        "candidate_chunks": candidate_chunks,
        "candidate_paths": candidate_paths,
        "path_lookup": path_lookup,
        "path_relation_candidates": path_relation_candidates,
        "new_entity_relation_candidates": new_entity_relation_candidates,
        "path_chunk_candidates": path_chunk_candidates,
        "new_entity_chunk_candidates": new_entity_chunk_candidates,
    }

print("Expansion functions are ready.")

Expansion functions are ready.


In [ ]:
# cell 26
# LLM selection and one-question traversal pipeline.

def select_hop23(question, already_chunks, candidate_chunks, candidate_paths):
    # Run HOP2/HOP3 LLM selector.
    already_chunks = dedupe_chunks_keep_best(already_chunks)
    candidate_chunks = dedupe_chunks_keep_best(candidate_chunks)
    candidate_paths = dedupe_paths_by_id(candidate_paths)

    already_prompt_chunks = [
        compact_chunk_for_prompt(item)
        for item in already_chunks
    ]

    candidate_prompt_chunks = [
        compact_chunk_for_prompt(item)
        for item in candidate_chunks
    ]

    prompt_paths, prompt_to_real, real_to_prompt = make_prompt_path_items(
        candidate_paths,
        compact_path_for_hop23_prompt,
        prefix="p",
    )

    prompt = render_prompt(
        HOP2_HOP3_PROMPT,
        question=question,
        selected_chunks=safe_json_dumps(already_prompt_chunks),
        candidate_chunks=safe_json_dumps(candidate_prompt_chunks),
        candidate_paths=safe_json_dumps(prompt_paths),
    )

    result = call_llm_json(
        prompt=prompt,
        schema=HOP2_HOP3_SCHEMA,
        max_tokens=HOP23_LLM_MAX_TOKENS,
        max_retries=2,
        print_final=False,
    )

    parsed = result["parsed"]

    chunk_lookup = {}

    for item in already_chunks + candidate_chunks:
        item = normalize_chunk_item(item)
        chunk_lookup[item["chunk_id"]] = item

    available_chunk_ids = [
        item["chunk_id"]
        for item in already_chunks + candidate_chunks
    ]

    selected_chunk_ids = sanitize_selected_chunk_ids(
        selected_ids=parsed.get("selected_chunk_ids", []),
        available_chunk_ids=available_chunk_ids,
        max_count=TRAVERSAL_CONFIG["max_final_chunks"],
    )

    selected_chunks = [
        chunk_lookup[chunk_id]
        for chunk_id in selected_chunk_ids
        if chunk_id in chunk_lookup
    ]

    selected_path_queries = sanitize_selected_path_queries(
        selected_path_queries=parsed.get("selected_path_queries", []),
        prompt_paths=prompt_paths,
        prompt_to_real=prompt_to_real,
        max_count=TRAVERSAL_CONFIG["max_final_paths"],
    )

    new_entity_queries = parsed.get("new_entity_queries", [])
    if not isinstance(new_entity_queries, list):
        new_entity_queries = []
    new_entity_queries = new_entity_queries[:2]

    path_lookup = {
        item["path_id"]: item
        for item in candidate_paths
    }

    return {
        "parsed": parsed,
        "selected_chunk_ids": selected_chunk_ids,
        "selected_chunks": selected_chunks,
        "selected_path_queries": selected_path_queries,
        "new_entity_queries": new_entity_queries,
        "candidate_paths": candidate_paths,
        "path_lookup": path_lookup,
        "prompt_paths": prompt_paths,
        "prompt_to_real": prompt_to_real,
    }

def select_hop4(question, already_chunks, candidate_chunks, candidate_paths):
    # Run final HOP4 LLM selector.
    already_chunks = dedupe_chunks_keep_best(already_chunks)
    candidate_chunks = dedupe_chunks_keep_best(candidate_chunks)
    candidate_paths = dedupe_paths_by_id(candidate_paths)

    already_prompt_chunks = [
        compact_chunk_for_prompt(item)
        for item in already_chunks
    ]

    candidate_prompt_chunks = [
        compact_chunk_for_prompt(item)
        for item in candidate_chunks
    ]

    prompt_paths, prompt_to_real, real_to_prompt = make_prompt_path_items(
        candidate_paths,
        compact_path_for_hop4_prompt,
        prefix="p",
    )

    prompt = render_prompt(
        HOP4_PROMPT,
        question=question,
        selected_chunks=safe_json_dumps(already_prompt_chunks),
        candidate_chunks=safe_json_dumps(candidate_prompt_chunks),
        candidate_paths=safe_json_dumps(prompt_paths),
    )

    result = call_llm_json(
        prompt=prompt,
        schema=HOP4_SCHEMA,
        max_tokens=HOP4_LLM_MAX_TOKENS,
        max_retries=2,
        print_final=False,
    )

    parsed = result["parsed"]

    chunk_lookup = {}

    for item in already_chunks + candidate_chunks:
        item = normalize_chunk_item(item)
        chunk_lookup[item["chunk_id"]] = item

    available_chunk_ids = [
        item["chunk_id"]
        for item in already_chunks + candidate_chunks
    ]

    selected_chunk_ids = sanitize_selected_chunk_ids(
        selected_ids=parsed.get("selected_chunk_ids", []),
        available_chunk_ids=available_chunk_ids,
        max_count=TRAVERSAL_CONFIG["max_final_chunks"],
    )

    selected_path_ids = sanitize_selected_path_ids(
        selected_path_ids=parsed.get("selected_path_ids", []),
        prompt_paths=prompt_paths,
        prompt_to_real=prompt_to_real,
        max_count=TRAVERSAL_CONFIG["max_final_paths"],
    )

    path_lookup = {
        item["path_id"]: item
        for item in candidate_paths
    }

    selected_chunks = [
        chunk_lookup[chunk_id]
        for chunk_id in selected_chunk_ids
        if chunk_id in chunk_lookup
    ]

    selected_paths = [
        path_lookup[path_id]
        for path_id in selected_path_ids
        if path_id in path_lookup
    ]

    return {
        "parsed": parsed,
        "selected_chunk_ids": selected_chunk_ids,
        "selected_chunks": selected_chunks,
        "selected_path_ids": selected_path_ids,
        "selected_paths": selected_paths,
        "candidate_paths": candidate_paths,
        "path_lookup": path_lookup,
        "prompt_paths": prompt_paths,
        "prompt_to_real": prompt_to_real,
    }

def format_evidence_chunk(chunk_item):
    # Output chunk with requested keys.
    chunk_item = normalize_chunk_item(chunk_item)
    return {
        "title": chunk_item["title"],
        "text": chunk_item["text"],
    }

def format_evidence_path(path_item):
    # Output path with chunk lineage.
    path_chunk_ids = path_item.get("path_chunk_ids", [])
    if not path_chunk_ids and path_item.get("chunk_id"):
        path_chunk_ids = [str(path_item["chunk_id"])]

    return {
        "path_id": path_item["path_id"],
        "path": path_item["readable_path"],
        "terminal_entity": path_item["terminal_entity"],
        "source_chunk_id": str(path_item["chunk_id"]),
        "path_depth": int(path_item.get("path_depth", 1)),
        "chunk_id": ordered_unique(path_chunk_ids),
    }

def make_success_record(record, final_chunks, final_paths):
    # Build successful output record.
    return {
        "source_index": int(record["source_index"]),
        "type": record.get("type"),
        "question": record.get("question"),
        "answer": record.get("answer"),
        "supports": record.get("supports", []),
        "evidence_chunk": [
            format_evidence_chunk(item)
            for item in final_chunks
        ],
        "evidence_path": [
            format_evidence_path(item)
            for item in final_paths
        ],
    }

def make_error_record(record, error):
    # Build error output record.
    return {
        "source_index": int(record["source_index"]),
        "type": record.get("type"),
        "question": record.get("question"),
        "answer": record.get("answer"),
        "supports": record.get("supports", []),
        "evidence_chunk": [],
        "evidence_path": [],
        "error": str(error),
        "traceback": traceback.format_exc(),
    }

def traverse_one_question(record):
    # Run the 3-LLM-call ablation for one question.
    question = record["question"]

    hop1 = run_hop1_retrieval(question)

    hop2 = select_hop23(
        question=question,
        already_chunks=[],
        candidate_chunks=hop1["selected_chunks"],
        candidate_paths=hop1["selected_paths"],
    )

    hop2_expansion = expand_after_hop(
        hop_label="hop2",
        selected_path_queries=hop2["selected_path_queries"],
        parent_path_lookup=hop1["path_lookup"],
        new_entity_queries=hop2["new_entity_queries"],
    )

    hop4 = select_hop4(
        question=question,
        already_chunks=hop2["selected_chunks"],
        candidate_chunks=hop2_expansion["candidate_chunks"],
        candidate_paths=hop2_expansion["candidate_paths"],
    )

    return make_success_record(
        record=record,
        final_chunks=hop4["selected_chunks"],
        final_paths=hop4["selected_paths"],
    )

print("One-question 3-LLM-call ablation pipeline is ready.")

One-question 3-LLM-call ablation pipeline is ready.


In [27]:
# cell 27
# Run traversal over all questions and save after each item.

existing_results = load_existing_results(PHASE2_OUTPUT_PATH)

start_idx = int(TRAVERSAL_START_INDEX)
end_idx = len(question_records) if TRAVERSAL_END_INDEX is None else int(TRAVERSAL_END_INDEX)

run_records = [
    rec
    for rec in question_records[start_idx:end_idx]
]

print("Total questions:", len(question_records))
print("Run range:", start_idx, "to", end_idx)
print("Existing saved records:", len(existing_results))
print("Output:", PHASE2_OUTPUT_PATH)

progress = tqdm(run_records, desc="Running 3-LLM-call ablation", dynamic_ncols=True)

for local_pos, record in enumerate(progress, start=1):
    source_index = int(record["source_index"])

    if source_index in existing_results and "error" not in existing_results[source_index]:
        progress.set_postfix({
            "source_index": source_index,
            "status": "skipped",
            "saved": len(existing_results),
        })
        continue

    try:
        output_record = traverse_one_question(record)
        existing_results[source_index] = output_record
        status = "ok"

    except Exception as e:
        output_record = make_error_record(record, e)
        existing_results[source_index] = output_record
        status = "error"

        gc.collect()
        torch.cuda.empty_cache()

    if local_pos % SAVE_EVERY_N_QUESTIONS == 0:
        save_result_map(PHASE2_OUTPUT_PATH, existing_results)

    if local_pos % CLEAR_CACHE_EVERY_N_QUESTIONS == 0:
        gc.collect()
        torch.cuda.empty_cache()

    progress.set_postfix({
        "source_index": source_index,
        "status": status,
        "saved": len(existing_results),
    })

save_result_map(PHASE2_OUTPUT_PATH, existing_results)

print("3-LLM-call ablation finished.")
print("Saved records:", len(existing_results))
print("Output file:", PHASE2_OUTPUT_PATH)

Total questions: 1000
Run range: 0 to 1000
Existing saved records: 0
Output: /content/drive/MyDrive/final_project/ablation/3llm_call_(hop)/2wikimultihopqa_dev_2020wiki_1000_traversal_evidence.json


Running 3-LLM-call ablation:   0%|          | 0/1000 [00:00<?, ?it/s]

3-LLM-call ablation finished.
Saved records: 1000
Output file: /content/drive/MyDrive/final_project/ablation/3llm_call_(hop)/2wikimultihopqa_dev_2020wiki_1000_traversal_evidence.json
